# Machine-Readable Reliability Passports for Medical Imaging AI
**Version 0.3 (schema 1.1.1).** Changes relative to version 0.2: every claim-gate record carries a `model_scope` naming the model it was computed from, together with a canonical digest of that model's test-set probability matrix, and conformal coverage is recorded once per model (C5 for the hierarchical model, C5b for the fine-tuned model) instead of being reported under a single unscoped gate.

**Version 0.2 (schema 1.1.0).** Changes relative to version 0.1: claim-gate records carry explicit prerequisite dependencies and threshold provenance; near-identity performance is reported per injected transformation stratum, gate C1 is keyed on the weakest stratum, and Wilson intervals are given only for non-deterministic strata; direct pairwise and same-component near-identity recall are both reported; conformal per-class calibration sizes and thresholds are exported; an end-to-end fine-tuned CNN is evaluated on the same locked partitions; 2,000 bootstrap replicates and a rule-of-three bound are used; a two-stage border-cropped near-duplicate detector is evaluated. Seed and partitions are unchanged.

## Colab-ready experimental notebook for *Informatics in Medicine Unlocked*

This notebook implements the experiments required by the manuscript:

1. a versioned JSON reliability-passport schema and evidence-gate engine;
2. a controlled fault-injection benchmark with known ground truth;
3. regeneration of the duplicate-aware three-class CXR audit;
4. optional regeneration of the inherited image-model reliability results;
5. perceptual-threshold sensitivity, repeated-run determinism, and scaling;
6. cross-modality portability on PathMNIST colorectal histopathology;
7. human-review candidate export without fabricating reviewer decisions;
8. manuscript-ready PNG figures, CSV tables, JSON evidence, and summaries.

The notebook writes all durable outputs to Google Drive. No manuscript value
is hard-coded as a new result: every reported result is computed during the run.



## 1. Environment, Google Drive, and prespecified configuration



In [ ]:
import importlib.util, subprocess, sys, os, json, random, hashlib, warnings, gc
import shutil, zipfile, time, tracemalloc, platform, re
from pathlib import Path

REQUIRED = {
    "imagehash": "ImageHash>=4.3.2",
    "timm": "timm>=1.0.19",
    "medmnist": "medmnist>=3.0.2",
    "jsonschema": "jsonschema>=4.23",
    "seaborn": "seaborn>=0.13.2",
    "psutil": "psutil>=5.9",
}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import imagehash, joblib, jsonschema, medmnist, psutil, timm, torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageEnhance, ImageOps
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from scipy.stats import norm
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, balanced_accuracy_score,
    confusion_matrix, f1_score, log_loss, precision_recall_fscore_support,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

warnings.filterwarnings("ignore", category=UserWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

CONFIG = {
    "seed": 20260816,
    "schema_version": "1.1.1",   # v0.3: model_scope on gate records (v0.2: gate dependencies + threshold provenance)
    "software_version": "0.3.0",
    "output_root": "/content/drive/MyDrive/Outputs/IMU_Reliability_Passport" if IN_COLAB else str(Path.cwd()/"IMU_Reliability_Passport_Outputs"),
    "cxr_dataset_root": "/content/drive/MyDrive/Datasets/Medical/CXR_Triage" if IN_COLAB else str(Path.cwd()/"kaggle_data"),
    "cxr_dataset_slug": "muhammadrehan00/chest-xray-dataset",
    "pathmnist_export_n": 9000,
    "fault_base_n": 90,
    "fault_negative_ratio": 2,
    "phash_radius": 4,
    "phash_radii": [0, 2, 4, 6, 8],
    "bootstrap_replicates": 2000,   # v0.2 (was 500)
    "run_reference_models": True,
    "encoder": "mobilenetv3_small_100",
    "image_size": 160,
    "batch_size": 128,
    "num_workers": 2,
    "conformal_alpha": 0.10,
    "target_auto_coverage": 0.75,
    "target_abnormal_sensitivity": 0.98,
    "target_tb_sensitivity": 0.95,
    "conformal_tolerance": 0.02,
    "shortcut_gap_tolerance": 0.05,
    "fault_min_recall": 0.95,
    "fault_max_fpr": 0.05,
    "scaling_sizes": [250, 500, 1000, 2500, 5000],
    "auto_download_archive": False,
    # v0.2 additions --------------------------------------------------------------
    "run_finetuned_model": True,
    "finetune": {"lr": 1e-4, "weight_decay": 1e-2, "batch_size": 64, "max_epochs": 20, "patience": 5,
                 "crop_scale": [0.9, 1.0], "hflip": True},
    "run_two_stage_detector": True,
    "two_stage": {"dev_base_n": 30, "candidate_radius": 8, "tau_grid": [0.80, 0.85, 0.90, 0.92, 0.94, 0.96, 0.98], "border_tol": 8},
    "gate_dependencies": {"C1": [], "C2": ["C1"], "C3": [], "C4": ["C2"], "C5": ["C2", "C4"],
                          "C5b": ["C2", "C4"], "C6": [], "C7": []},   # v0.3: C5b = conformal coverage of the fine-tuned model
}

SEED = CONFIG["seed"]
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

ROOT = Path(CONFIG["output_root"])
FIG_DIR = ROOT / "Figures"
TAB_DIR = ROOT / "Tables"
RAW_DIR = ROOT / "Raw"
MODEL_DIR = ROOT / "Models"
CACHE_DIR = ROOT / "Cache"
FAULT_DIR = ROOT / "FaultBenchmark"
PORT_DIR = ROOT / "Portability_PathMNIST"
REVIEW_DIR = ROOT / "HumanReview"
COMPARE_DIR = ROOT / "ComparisonRuns"
for directory in (ROOT, FIG_DIR, TAB_DIR, RAW_DIR, MODEL_DIR, CACHE_DIR, FAULT_DIR, PORT_DIR, REVIEW_DIR, COMPARE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = ROOT / "Outputs_Summary.txt"
LOG_PATH.write_text("", encoding="utf-8")
def log(message=""):
    print(message, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as stream:
        stream.write(str(message) + "\n")

environment = {
    "python": sys.version.split()[0], "platform": platform.platform(),
    "torch": torch.__version__, "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpu_count": os.cpu_count(), "ram_gb": round(psutil.virtual_memory().total/2**30, 2),
}
environment["fingerprint"] = hashlib.sha256(json.dumps(environment, sort_keys=True).encode()).hexdigest()
(RAW_DIR/"configuration.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
(RAW_DIR/"environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
log(json.dumps({"environment": environment, "configuration": CONFIG}, indent=2))



## 2. Dataset acquisition and adapters

The CXR adapter follows the supplied notebook. PathMNIST is deliberately used
as the second dataset because it changes both organization and modality
(colorectal histopathology rather than radiography).



In [ ]:
CLASS_NAMES = ["Normal", "Pneumonia", "Tuberculosis"]
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
FOLDER_TO_CLASS = {name.lower(): name for name in CLASS_NAMES}

def valid_cxr_root(root):
    root = Path(root)
    return all((root/split/label).is_dir() for split in ("train","val","test") for label in FOLDER_TO_CLASS)

def locate_cxr_root(base):
    base = Path(base)
    candidates = [base]
    if base.exists(): candidates += [p.parent for p in base.rglob("train") if p.is_dir()]
    for candidate in candidates:
        if valid_cxr_root(candidate): return candidate.resolve()
    return None

requested_cxr = Path(CONFIG["cxr_dataset_root"])
CXR_ROOT = locate_cxr_root(requested_cxr)
if CXR_ROOT is None:
    log("CXR data not found in Drive; downloading through KaggleHub.")
    if importlib.util.find_spec("kagglehub") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub>=0.3.12"])
    import kagglehub
    partial = requested_cxr.exists() and any(requested_cxr.iterdir())   # incomplete earlier download
    downloaded = kagglehub.dataset_download(CONFIG["cxr_dataset_slug"], output_dir=str(requested_cxr), force_download=partial)
    CXR_ROOT = locate_cxr_root(requested_cxr) or locate_cxr_root(downloaded)
if CXR_ROOT is None:
    raise FileNotFoundError("Expected CXR train/val/test and normal/pneumonia/tuberculosis folders were not found.")

cxr_records = []
for source_split in ("train", "val", "test"):
    for folder, class_name in FOLDER_TO_CLASS.items():
        for path in sorted((CXR_ROOT/source_split/folder).glob("*.jpg")):
            cxr_records.append({
                "record_id": f"cxr_{len(cxr_records):07d}", "path": str(path),
                "dataset": "CXR compilation", "source_split": source_split,
                "class_name": class_name, "label": CLASS_TO_INDEX[class_name],
                "declared_width": np.nan, "declared_height": np.nan,
            })
cxr_index = pd.DataFrame(cxr_records)
log(f"CXR adapter: {len(cxr_index):,} files from {CXR_ROOT}")

def export_pathmnist(max_n):
    index_file = PORT_DIR/"pathmnist_index.csv"
    if index_file.exists():
        existing = pd.read_csv(index_file)
        if existing.path.map(lambda p: Path(p).exists()).all(): return existing
    info = medmnist.INFO["pathmnist"]
    DataClass = getattr(medmnist, info["python_class"])
    records=[]
    per_split=max(1, max_n//3)
    for split in ("train","val","test"):
        data=DataClass(split=split, download=True, size=28)
        labels=np.asarray(data.labels).reshape(-1)
        rng=np.random.default_rng(SEED+{"train":0,"val":1,"test":2}[split])
        chosen=[]
        for label in sorted(np.unique(labels)):
            ids=np.flatnonzero(labels==label)
            take=min(len(ids), max(1, per_split//len(np.unique(labels))))
            chosen.extend(rng.choice(ids, take, replace=False).tolist())
        chosen=chosen[:per_split]
        split_dir=PORT_DIR/split; split_dir.mkdir(parents=True,exist_ok=True)
        for idx in chosen:
            image,label=data[idx]; label=int(np.asarray(label).reshape(-1)[0])
            path=split_dir/f"pathmnist_{split}_{idx:06d}.png"
            if not path.exists(): image.convert("RGB").save(path)
            records.append({"record_id":f"path_{split}_{idx:06d}","path":str(path),
                "dataset":"PathMNIST","source_split":split,"class_name":info["label"][str(label)],
                "label":label,"declared_width":28,"declared_height":28})
    frame=pd.DataFrame(records);frame.to_csv(index_file,index=False);return frame

pathmnist_index = export_pathmnist(CONFIG["pathmnist_export_n"])
log(f"PathMNIST portability adapter: {len(pathmnist_index):,} exported images")



## 3. Core passport audit engine

Exact byte hashes, decoded-pixel hashes, perceptual hashes, native image
statistics, connected duplicate groups, label conflicts, split leakage, and
metadata completeness are calculated by the same functions for both datasets.



In [ ]:
from concurrent.futures import ThreadPoolExecutor

def sha256_bytes(data): return hashlib.sha256(data).hexdigest()

def audit_one(record):
    out=dict(record); path=Path(record["path"])
    out.update({"readable":False,"sha256":"","pixel_sha256":"","phash":"",
                "width":np.nan,"height":np.nan,"mode":"","mean":np.nan,"std":np.nan,
                "sharpness":np.nan,"dark_fraction":np.nan,"bright_fraction":np.nan,"error":""})
    try:
        raw=path.read_bytes()
        with Image.open(path) as image:
            image.load(); rgb=image.convert("RGB"); gray=image.convert("L")
            arr_rgb=np.asarray(rgb); thumb=gray.copy(); thumb.thumbnail((192,192),Image.Resampling.BILINEAR)
            a=np.asarray(thumb,dtype=np.float32)/255.0
            lap=-4*a+np.roll(a,1,0)+np.roll(a,-1,0)+np.roll(a,1,1)+np.roll(a,-1,1)
            pixel_payload=f"RGB|{rgb.width}|{rgb.height}|".encode()+arr_rgb.tobytes()
            out.update({"readable":True,"sha256":sha256_bytes(raw),"pixel_sha256":sha256_bytes(pixel_payload),
                "phash":str(imagehash.phash(gray)),"width":rgb.width,"height":rgb.height,"mode":image.mode,
                "mean":float(a.mean()),"std":float(a.std()),"sharpness":float(lap.var()),
                "dark_fraction":float((a<.02).mean()),"bright_fraction":float((a>.98).mean())})
    except Exception as exc: out["error"]=repr(exc)
    declared_ok=True
    for key in ("declared_width","declared_height"):
        value=record.get(key,np.nan)
        if pd.notna(value) and (not np.isfinite(float(value)) or float(value)<=0): declared_ok=False
    out["metadata_valid"] = bool(declared_ok and record.get("metadata_corrupt",False) is not True)
    return out

class UnionFind:
    def __init__(self,n): self.parent=list(range(n));self.rank=[0]*n
    def find(self,x):
        while self.parent[x]!=x:
            self.parent[x]=self.parent[self.parent[x]];x=self.parent[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1

class BKNode:
    def __init__(self,value,index):self.value=value;self.indices=[index];self.children={}
class BKTree:
    def __init__(self):self.root=None
    def add(self,value,index):
        if self.root is None:self.root=BKNode(value,index);return
        node=self.root
        while True:
            distance=(value^node.value).bit_count()
            if distance==0:node.indices.append(index);return
            if distance not in node.children:node.children[distance]=BKNode(value,index);return
            node=node.children[distance]
    def query(self,value,radius):
        if self.root is None:return []
        found=[];stack=[self.root]
        while stack:
            node=stack.pop();distance=(value^node.value).bit_count()
            if distance<=radius:found.extend(node.indices)
            stack.extend(child for edge,child in node.children.items() if distance-radius<=edge<=distance+radius)
        return found

def group_from_values(values):
    codes,_=pd.factorize(pd.Series(values),sort=True)
    return codes.astype(int)

def phash_groups(hex_hashes,radius):
    values=[int(str(v),16) for v in hex_hashes];uf=UnionFind(len(values));tree=BKTree()
    for i,value in enumerate(values):
        for neighbor in tree.query(value,radius):uf.union(i,neighbor)
        tree.add(value,i)
    roots=[uf.find(i) for i in range(len(values))]
    mapping={root:k for k,root in enumerate(sorted(set(roots)))}
    return np.array([mapping[root] for root in roots],dtype=int)

def enrich_groups(audit_df,radius):
    df=audit_df.loc[audit_df.readable].copy().reset_index(drop=True)
    df["exact_group"]=group_from_values(df.sha256)
    df["pixel_group"]=group_from_values(df.pixel_sha256)
    df["phash_group"]=phash_groups(df.phash.tolist(),radius)
    for prefix in ("exact","pixel","phash"):
        group=f"{prefix}_group"
        sizes=df.groupby(group).size();classes=df.groupby(group).class_name.nunique();splits=df.groupby(group).source_split.nunique()
        df[f"{prefix}_group_size"]=df[group].map(sizes).astype(int)
        df[f"{prefix}_label_conflict"]=df[group].map(classes).gt(1)
        df[f"{prefix}_split_leakage"]=df[group].map(splits).gt(1)
    return df

def audit_manifest(index_df,radius,workers=4,progress_label="audit"):
    completed=[]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for i,result in enumerate(pool.map(audit_one,index_df.to_dict("records")),1):
            completed.append(result)
            if i%1000==0 or i==len(index_df):log(f"{progress_label}: {i:,}/{len(index_df):,}")
    return enrich_groups(pd.DataFrame(completed),radius)

def canonical_digest(df,columns):
    text=df[columns].sort_values(columns[0]).to_csv(index=False,float_format="%.12g",lineterminator="\n")
    return sha256_bytes(text.encode())

def root_manifest_hash(df):
    return canonical_digest(df,["record_id","sha256","pixel_sha256","class_name","source_split"])



## 4. Real-world CXR integrity passport and group-aware partitions



In [ ]:
cxr_cache=CACHE_DIR/"cxr_audit_radius4.csv"
if cxr_cache.exists():
    cxr_audit=pd.read_csv(cxr_cache)
    for col in [c for c in cxr_audit if c.endswith(("_conflict","_leakage")) or c in ("readable","metadata_valid")]:
        cxr_audit[col]=cxr_audit[col].astype(bool)
    log(f"Loaded cached CXR audit: {len(cxr_audit):,}")
else:
    cxr_audit=audit_manifest(cxr_index,CONFIG["phash_radius"],progress_label="CXR audit")
    cxr_audit.to_csv(cxr_cache,index=False)

exact_summary=cxr_audit.groupby("sha256").agg(n=("record_id","size"),n_classes=("class_name","nunique")).reset_index()
conflict_hashes=set(exact_summary.loc[exact_summary.n_classes>1,"sha256"])
exact_conflicts=cxr_audit.loc[cxr_audit.sha256.isin(conflict_hashes)].copy()
unique_cxr=(cxr_audit.loc[~cxr_audit.sha256.isin(conflict_hashes)].sort_values(["sha256","path"])
            .drop_duplicates("sha256").reset_index(drop=True))
unique_cxr["near_group"]=phash_groups(unique_cxr.phash.tolist(),CONFIG["phash_radius"])
near_summary=unique_cxr.groupby("near_group").agg(n=("record_id","size"),n_classes=("class_name","nunique")).reset_index()
near_conflict_groups=set(near_summary.loc[near_summary.n_classes>1,"near_group"])
near_conflicts=unique_cxr.loc[unique_cxr.near_group.isin(near_conflict_groups)].copy()
clean_cxr=unique_cxr.loc[~unique_cxr.near_group.isin(near_conflict_groups)].copy().reset_index(drop=True)
clean_cxr["label"]=clean_cxr.class_name.map(CLASS_TO_INDEX).astype(int)
clean_cxr["duplicate_group"]=clean_cxr.near_group.astype(int)

def holdout(frame,n_splits,seed):
    splitter=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    keep,take=next(splitter.split(frame,frame.label,frame.duplicate_group))
    return frame.iloc[keep].reset_index(drop=True),frame.iloc[take].reset_index(drop=True)

remaining,test_df=holdout(clean_cxr,10,SEED)
remaining,calibration_df=holdout(remaining,9,SEED+1)
train_df,val_df=holdout(remaining,8,SEED+2)
cxr_frames={"train":train_df,"validation":val_df,"calibration":calibration_df,"test":test_df}
cxr_assignment=pd.concat([f.assign(experimental_split=name) for name,f in cxr_frames.items()],ignore_index=True)
assert cxr_assignment.groupby("sha256").experimental_split.nunique().max()==1
assert cxr_assignment.groupby("duplicate_group").experimental_split.nunique().max()==1

for name,frame in cxr_frames.items():frame.to_csv(TAB_DIR/f"CXR_split_{name}.csv",index=False)
exact_conflicts.to_csv(TAB_DIR/"CXR_excluded_exact_label_conflicts.csv",index=False)
near_conflicts.to_csv(TAB_DIR/"CXR_excluded_near_label_conflicts.csv",index=False)

cxr_integrity={
    "source_files":int(len(cxr_audit)),"unique_sha256":int(cxr_audit.sha256.nunique()),
    "exact_conflict_identities":int(len(conflict_hashes)),"exact_conflict_files":int(len(exact_conflicts)),
    "near_conflict_groups":int(len(near_conflict_groups)),"near_conflict_images":int(len(near_conflicts)),
    "final_clean_images":int(len(clean_cxr)),"root_manifest_hash":root_manifest_hash(cxr_audit),
    "class_counts":{k:int(v) for k,v in clean_cxr.class_name.value_counts().to_dict().items()},
    "split_counts":{k:int(v) for k,v in cxr_assignment.experimental_split.value_counts().to_dict().items()},
}
(RAW_DIR/"CXR_integrity_audit.json").write_text(json.dumps(cxr_integrity,indent=2),encoding="utf-8")
log("CXR integrity evidence:\n"+json.dumps(cxr_integrity,indent=2))



## 5. Controlled fault-injection generator

The generator and audit engine are isolated. Ground truth is written before
detector evaluation. Each base image receives controlled transformations,
label conflict, split leakage, source artifact, and metadata corruption.



In [ ]:
def stratified_base_sample(frame,n):
    parts=[];per=max(1,n//frame.class_name.nunique())
    for _,group in frame.groupby("class_name"):
        parts.append(group.sample(min(per,len(group)),random_state=SEED))
    return pd.concat(parts).head(n).reset_index(drop=True)

def changed_class(name):
    names=sorted(CLASS_NAMES)
    return names[(names.index(name)+1)%len(names)]

def safe_save(image,path,fmt="PNG",**kwargs):
    path.parent.mkdir(parents=True,exist_ok=True);image.save(path,format=fmt,**kwargs)

def build_fault_benchmark(base_frame):
    index_file=FAULT_DIR/"fault_index.csv";truth_file=FAULT_DIR/"fault_ground_truth.csv";pairs_file=FAULT_DIR/"pair_ground_truth.csv"
    if index_file.exists() and truth_file.exists() and pairs_file.exists():
        return pd.read_csv(index_file),pd.read_csv(truth_file),pd.read_csv(pairs_file)
    image_dir=FAULT_DIR/"images";image_dir.mkdir(parents=True,exist_ok=True)
    records=[];truth=[];pairs=[]
    for i,row in base_frame.reset_index(drop=True).iterrows():
        base_id=f"base_{i:04d}";base_path=image_dir/f"{base_id}.png"
        with Image.open(row.path) as src:
            image=src.convert("RGB");safe_save(image,base_path)
            w,h=image.size
            base={"record_id":base_id,"base_id":base_id,"path":str(base_path),"dataset":"Controlled benchmark",
                  "source_split":"train","class_name":row.class_name,"label":CLASS_TO_INDEX[row.class_name],
                  "declared_width":w,"declared_height":h,"metadata_corrupt":False,"fault_type":"control"}
            records.append(base);truth.append({**base,"expected_metadata_fault":0})
            variants=[]
            exact_path=image_dir/f"{base_id}_exact.png";shutil.copy2(base_path,exact_path)
            variants.append(("exact_duplicate",exact_path,row.class_name,"train",w,h,False))
            pixel_path=image_dir/f"{base_id}_pixel.bmp";safe_save(image,pixel_path,fmt="BMP")
            variants.append(("pixel_reencode",pixel_path,row.class_name,"train",w,h,False))
            resize_path=image_dir/f"{base_id}_resize.png"
            small=image.resize((max(16,int(w*.82)),max(16,int(h*.82))),Image.Resampling.BICUBIC)
            safe_save(small.resize((w,h),Image.Resampling.BICUBIC),resize_path)
            variants.append(("near_resize",resize_path,row.class_name,"train",w,h,False))
            contrast_path=image_dir/f"{base_id}_contrast.png";safe_save(ImageEnhance.Contrast(image).enhance(1.10),contrast_path)
            variants.append(("near_contrast",contrast_path,row.class_name,"train",w,h,False))
            jpeg_path=image_dir/f"{base_id}_compression.jpg";safe_save(image,jpeg_path,fmt="JPEG",quality=55,optimize=True)
            variants.append(("near_compression",jpeg_path,row.class_name,"train",w,h,False))
            border_path=image_dir/f"{base_id}_border.png";pad=max(3,min(w,h)//40)
            bordered=ImageOps.expand(image,border=pad,fill="white").resize((w,h),Image.Resampling.BILINEAR)
            safe_save(bordered,border_path)
            variants.append(("source_border",border_path,row.class_name,"train",w,h,False))
            conflict_path=image_dir/f"{base_id}_label_conflict.png";shutil.copy2(base_path,conflict_path)
            variants.append(("label_conflict",conflict_path,changed_class(row.class_name),"train",w,h,False))
            leak_path=image_dir/f"{base_id}_split_leak.png";shutil.copy2(base_path,leak_path)
            variants.append(("split_leakage",leak_path,row.class_name,"test",w,h,False))
            meta_path=image_dir/f"{base_id}_metadata.png";safe_save(ImageEnhance.Brightness(image).enhance(0.985),meta_path)
            variants.append(("metadata_fault",meta_path,row.class_name,"train",-1,np.nan,True))
        for fault_type,path,class_name,split,dw,dh,corrupt in variants:
            rid=f"{base_id}_{fault_type}"
            rec={"record_id":rid,"base_id":base_id,"path":str(path),"dataset":"Controlled benchmark",
                 "source_split":split,"class_name":class_name,"label":CLASS_TO_INDEX[class_name],
                 "declared_width":dw,"declared_height":dh,"metadata_corrupt":corrupt,"fault_type":fault_type}
            records.append(rec);truth.append({**rec,"expected_metadata_fault":int(fault_type=="metadata_fault")})
            pairs.append({"base_record_id":base_id,"other_record_id":rid,"fault_type":fault_type,"is_related":1})
    index=pd.DataFrame(records);ground=pd.DataFrame(truth);pair_truth=pd.DataFrame(pairs)
    bases=index.loc[index.fault_type=="control","record_id"].tolist();rng=np.random.default_rng(SEED)
    positives=len(pair_truth)
    negatives=[]
    while len(negatives)<CONFIG["fault_negative_ratio"]*positives:
        a,b=rng.choice(bases,2,replace=False)
        negatives.append({"base_record_id":a,"other_record_id":b,"fault_type":"unrelated_control","is_related":0})
    pair_truth=pd.concat([pair_truth,pd.DataFrame(negatives)],ignore_index=True)
    index.to_csv(index_file,index=False);ground.to_csv(truth_file,index=False);pair_truth.to_csv(pairs_file,index=False)
    return index,ground,pair_truth

fault_bases=stratified_base_sample(clean_cxr,CONFIG["fault_base_n"])
fault_index,fault_truth,pair_truth=build_fault_benchmark(fault_bases)
log(f"Controlled benchmark: {len(fault_bases)} bases, {len(fault_index)} records, {len(pair_truth)} evaluated pairs")



## 6. Controlled fault-detection performance



In [ ]:
fault_audit=audit_manifest(fault_index,CONFIG["phash_radius"],progress_label="fault audit")
fault_audit.to_csv(TAB_DIR/"controlled_fault_audit.csv",index=False)
lookup=fault_audit.set_index("record_id")

def same_group(a,b,column):return int(lookup.loc[a,column]==lookup.loc[b,column])
pair_eval=pair_truth.copy()
pair_eval["exact_pred"]=[same_group(a,b,"exact_group") for a,b in zip(pair_eval.base_record_id,pair_eval.other_record_id)]
pair_eval["pixel_pred"]=[same_group(a,b,"pixel_group") for a,b in zip(pair_eval.base_record_id,pair_eval.other_record_id)]
pair_eval["near_pred"]=[same_group(a,b,"phash_group") for a,b in zip(pair_eval.base_record_id,pair_eval.other_record_id)]
exact_faults={"exact_duplicate","label_conflict","split_leakage"}
pixel_faults=exact_faults|{"pixel_reencode"}
# v0.2: the brightness-altered metadata-fault image is derived from the same base image and is therefore a
# near-identity positive (corrected reference endpoint).
near_faults=pixel_faults|{"near_resize","near_contrast","near_compression","source_border","metadata_fault"}
pair_eval["exact_true"]=pair_eval.fault_type.isin(exact_faults).astype(int)
pair_eval["pixel_true"]=pair_eval.fault_type.isin(pixel_faults).astype(int)
pair_eval["near_true"]=pair_eval.fault_type.isin(near_faults).astype(int)

def wilson_interval(success,n,alpha=.05):
    if n==0:return (np.nan,np.nan)
    z=norm.ppf(1-alpha/2);p=success/n;den=1+z*z/n
    center=(p+z*z/(2*n))/den;half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0,center-half),min(1,center+half)

def binary_row(name,y_true,y_pred,transformation):
    y_true=np.asarray(y_true,dtype=int);y_pred=np.asarray(y_pred,dtype=int)
    tp=int(((y_true==1)&(y_pred==1)).sum());fn=int(((y_true==1)&(y_pred==0)).sum())
    fp=int(((y_true==0)&(y_pred==1)).sum());tn=int(((y_true==0)&(y_pred==0)).sum())
    recall=tp/max(tp+fn,1);fpr=fp/max(fp+tn,1);lo,hi=wilson_interval(tp,tp+fn)
    return {"fault_type":name,"transformation":transformation,"injected_n":int((y_true==1).sum()),
        "precision":precision_score(y_true,y_pred,zero_division=0),"recall":recall,
        "recall_ci_low":lo,"recall_ci_high":hi,"f1":f1_score(y_true,y_pred,zero_division=0),
        "false_positive_rate":fpr,"tp":tp,"fp":fp,"fn":fn,"tn":tn}

results=[]
results.append(binary_row("Exact duplicate",pair_eval.exact_true,pair_eval.exact_pred,"Byte-identical copies"))
results.append(binary_row("Pixel identity",pair_eval.pixel_true,pair_eval.pixel_pred,"Lossless re-encoding plus exact copies"))
results.append(binary_row("Near duplicate",pair_eval.near_true,pair_eval.near_pred,"Resize, contrast, compression, and border"))

candidate=fault_audit.loc[fault_audit.fault_type.isin(["label_conflict","exact_duplicate","split_leakage"])].copy()
pair_classes_a=pair_eval.base_record_id.map(lookup.class_name);pair_classes_b=pair_eval.other_record_id.map(lookup.class_name)
label_conflict_pred=pair_eval.exact_pred.astype(bool)&pair_classes_a.ne(pair_classes_b)
results.append(binary_row("Label conflict",pair_eval.fault_type.eq("label_conflict"),label_conflict_pred,"Matched identity with altered label"))
pair_split_a=pair_eval.base_record_id.map(lookup.source_split);pair_split_b=pair_eval.other_record_id.map(lookup.source_split)
leakage_pred=pair_eval.exact_pred.astype(bool)&pair_split_a.ne(pair_split_b)
results.append(binary_row("Split leakage",pair_eval.fault_type.eq("split_leakage"),leakage_pred,"Matched identity across partitions"))
candidate=fault_audit.loc[fault_audit.fault_type.isin(["metadata_fault","control"])].copy()
results.append(binary_row("Metadata fault",candidate.fault_type.eq("metadata_fault"),~candidate.metadata_valid,"Missing or invalid declared metadata"))

fault_metrics=pd.DataFrame(results)
# v0.2: exact-match operators on fixed inputs have no sampling distribution; the pooled near-identity
# recall is a design-weighted mean over nine strata of 90 pairs. Wilson columns are therefore not exported for Table 2.
fault_metrics.drop(columns=["recall_ci_low","recall_ci_high"]).to_csv(TAB_DIR/"Table2_Controlled_Fault_Injection.csv",index=False)

# v0.2: per-stratum near-identity recall; Wilson interval only for non-deterministic strata.
pos=pair_eval.loc[pair_eval.near_true.eq(1)].copy()
stratum_rows=[]
for fault_type,group in pos.groupby("fault_type"):
    tp=int(group.near_pred.sum());n=int(len(group));lo,hi=wilson_interval(tp,n)
    deterministic=tp in (0,n)
    stratum_rows.append({"fault_type":fault_type,"injected_n":n,"detected":tp,"recall":tp/n,
        "wilson_low":np.nan if deterministic else lo,"wilson_high":np.nan if deterministic else hi,"deterministic":deterministic})
near_strata=pd.DataFrame(stratum_rows).sort_values("recall")
near_strata.to_csv(TAB_DIR/"Near_Identity_Stratum_Recall_radius4.csv",index=False)
min_stratum_recall=float(near_strata.recall.min());min_stratum=str(near_strata.iloc[0].fault_type)
p_var=near_strata.loc[~near_strata.deterministic]
design_se=float(np.sqrt(sum((1/len(near_strata))**2*r.recall*(1-r.recall)/r.injected_n for _,r in p_var.iterrows()))) if len(p_var) else 0.0
log(f"Near-identity strata (radius {CONFIG['phash_radius']}):\n"+near_strata.round(4).to_string(index=False))
log(f"Design-weighted pooled recall {pos.near_pred.mean():.4f}; design-based SE {design_se:.4f}; minimum stratum {min_stratum} = {min_stratum_recall:.4f}")

# v0.2: direct pairwise recall versus same-component recall, with the transitive fraction.
def hamming_hex(a,b):return (int(str(a),16)^int(str(b),16)).bit_count()
pair_eval["near_direct"]=[int(hamming_hex(lookup.loc[a,"phash"],lookup.loc[b,"phash"])<=CONFIG["phash_radius"])
                          for a,b in zip(pair_eval.base_record_id,pair_eval.other_record_id)]
pos=pair_eval.loc[pair_eval.near_true.eq(1)]
direct_vs_component={"same_component_recall":float(pos.near_pred.mean()),"direct_pairwise_recall":float(pos.near_direct.mean()),
    "transitive_fraction":float(((pos.near_pred==1)&(pos.near_direct==0)).sum()/max(int((pos.near_pred==1).sum()),1)),
    "unrelated_direct_positives":int(pair_eval.loc[pair_eval.near_true.eq(0),"near_direct"].sum())}
by_stratum=pos.groupby("fault_type")[["near_pred","near_direct"]].mean().rename(columns={"near_pred":"same_component_recall","near_direct":"direct_pairwise_recall"})
by_stratum["transitive_fraction"]=pos.groupby("fault_type").apply(lambda g:((g.near_pred==1)&(g.near_direct==0)).sum()/max(int((g.near_pred==1).sum()),1))
by_stratum.to_csv(TAB_DIR/"Near_Identity_Direct_vs_Component.csv")
(RAW_DIR/"near_identity_direct_vs_component.json").write_text(json.dumps(direct_vs_component,indent=2),encoding="utf-8")
log("Direct vs. same-component near-identity recall:\n"+json.dumps(direct_vs_component,indent=2)+"\n"+by_stratum.round(4).to_string())
pair_eval.to_csv(TAB_DIR/"controlled_pair_predictions.csv",index=False)
log("Controlled fault performance:\n"+fault_metrics.round(4).to_string(index=False))



## 7. Perceptual-threshold sensitivity and controlled source shortcut



In [ ]:
threshold_rows=[]
for radius in CONFIG["phash_radii"]:
    groups=phash_groups(fault_audit.phash.tolist(),radius)
    temp=fault_audit[["record_id"]].copy();temp["group"]=groups;group_map=temp.set_index("record_id").group
    pred=np.array([int(group_map[a]==group_map[b]) for a,b in zip(pair_eval.base_record_id,pair_eval.other_record_id)])
    true=pair_eval.near_true.to_numpy()
    row=binary_row("Near duplicate",true,pred,f"pHash radius {radius}")
    row.update({"radius":radius,"n_components":int(len(np.unique(groups))),
                "largest_component":int(pd.Series(groups).value_counts().max())})
    threshold_rows.append(row)
threshold_sensitivity=pd.DataFrame(threshold_rows).drop(columns=["recall_ci_low","recall_ci_high"])  # v0.2: no pooled Wilson interval
threshold_sensitivity.to_csv(TAB_DIR/"Perceptual_Threshold_Sensitivity.csv",index=False)

# Synthetic source-artifact control: label association is intentionally injected.
source_subset=fault_audit.loc[fault_audit.fault_type.isin(["control","source_border"])].copy()
source_subset["synthetic_endpoint"]=(source_subset.fault_type=="source_border").astype(int)
meta_cols=["width","height","mean","std","sharpness","dark_fraction","bright_fraction"]
rng=np.random.default_rng(SEED);order=rng.permutation(len(source_subset));cut=int(.7*len(order));tr,te=order[:cut],order[cut:]
source_model=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=500,random_state=SEED))
source_model.fit(source_subset.iloc[tr][meta_cols],source_subset.iloc[tr].synthetic_endpoint)
source_pred=source_model.predict(source_subset.iloc[te][meta_cols])
source_control={"accuracy":accuracy_score(source_subset.iloc[te].synthetic_endpoint,source_pred),
                "balanced_accuracy":balanced_accuracy_score(source_subset.iloc[te].synthetic_endpoint,source_pred),
                "f1":f1_score(source_subset.iloc[te].synthetic_endpoint,source_pred)}
pd.DataFrame([source_control]).to_csv(TAB_DIR/"Controlled_Source_Artifact_Diagnostic.csv",index=False)
log("Threshold sensitivity:\n"+threshold_sensitivity.round(4).to_string(index=False))



## 7b. Two-stage border-cropped near-duplicate detector (v0.2)

Stage 1 generates candidate pairs by perceptual hash of border-cropped images at Hamming radius eight; stage 2 accepts a candidate when the cosine similarity between frozen MobileNetV3-Small embeddings of the cropped images is at least τ. τ is selected on a separate development benchmark (30 base images disjoint from the 90 benchmark bases) as the largest value that maximises the minimum stratum recall subject to the false-positive ceiling; the frozen 90-base benchmark is evaluated once.


In [ ]:
two_stage_results={"status":"not_evaluated"}
if CONFIG["run_two_stage_detector"]:
    TS=CONFIG["two_stage"]
    def trim_uniform_border(image,tol=TS["border_tol"]):
        a=np.asarray(image.convert("L"),dtype=np.int16);h,w=a.shape
        uniform=lambda v:int(v.max()-v.min())<=tol
        top=0
        while top<h-1 and uniform(a[top]):top+=1
        bottom=h
        while bottom>top+1 and uniform(a[bottom-1]):bottom-=1
        left=0
        while left<w-1 and uniform(a[:,left]):left+=1
        right=w
        while right>left+1 and uniform(a[:,right-1]):right-=1
        return image.crop((left,top,right,bottom))

    ts_encoder=timm.create_model(CONFIG["encoder"],pretrained=True,num_classes=0,global_pool="avg").to(DEVICE).eval()
    for p in ts_encoder.parameters():p.requires_grad=False
    ts_transform=transforms.Compose([transforms.Resize((CONFIG["image_size"],CONFIG["image_size"]),interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

    def two_stage_features(index_df):
        hashes=[];embs=[];ids=[]
        batch=[];batch_ids=[]
        def flush():
            if not batch:return
            with torch.inference_mode():x=torch.stack(batch).to(DEVICE);e=ts_encoder(x).float().cpu().numpy()
            embs.append(e);ids.extend(batch_ids);batch.clear();batch_ids.clear()
        for row in index_df.itertuples():
            with Image.open(row.path) as img:
                cropped=trim_uniform_border(img.convert("RGB"))
                hashes.append(int(str(imagehash.phash(cropped.convert("L"))),16))
                batch.append(ts_transform(cropped));batch_ids.append(row.record_id)
            if len(batch)>=CONFIG["batch_size"]:flush()
        flush()
        E=np.concatenate(embs);E/=np.linalg.norm(E,axis=1,keepdims=True)+1e-12
        return pd.DataFrame({"record_id":ids,"phash_crop":hashes}),E

    def two_stage_predict(feat,E,pairs_df,tau,radius=TS["candidate_radius"]):
        n=len(feat);pos={rid:i for i,rid in enumerate(feat.record_id)};h=feat.phash_crop.to_numpy(dtype=object)
        uf=UnionFind(n);direct=np.zeros((n,n),dtype=bool)
        sims=E@E.T
        for i in range(n):
            for j in range(i+1,n):
                if (int(h[i])^int(h[j])).bit_count()<=radius and sims[i,j]>=tau:
                    uf.union(i,j);direct[i,j]=direct[j,i]=True
        roots=np.array([uf.find(i) for i in range(n)])
        comp=np.array([int(roots[pos[a]]==roots[pos[b]]) for a,b in zip(pairs_df.base_record_id,pairs_df.other_record_id)])
        dire=np.array([int(direct[pos[a],pos[b]]) for a,b in zip(pairs_df.base_record_id,pairs_df.other_record_id)])
        return comp,dire

    def two_stage_score(pairs_df,comp):
        truth=pairs_df.fault_type.isin(near_faults).astype(int).to_numpy()
        fpr=float(comp[truth==0].mean()) if (truth==0).any() else 0.0
        strata={ft:float(comp[(truth==1)&(pairs_df.fault_type==ft).to_numpy()].mean()) for ft in sorted(near_faults)}
        return {"pooled_recall":float(comp[truth==1].mean()),"min_stratum_recall":min(strata.values()),"fpr":fpr,
                "fp_count":int(comp[truth==0].sum()),"n_controls":int((truth==0).sum()),"strata":strata}

    # Development benchmark: 30 base images disjoint from the 90 frozen benchmark bases.
    dev_pool=clean_cxr.loc[~clean_cxr.record_id.isin(set(fault_bases.record_id))].reset_index(drop=True)
    dev_bases=stratified_base_sample(dev_pool,TS["dev_base_n"])
    _FAULT_DIR_MAIN=FAULT_DIR;FAULT_DIR=ROOT/"FaultBenchmark_Dev";FAULT_DIR.mkdir(parents=True,exist_ok=True)
    dev_index,dev_truth,dev_pairs=build_fault_benchmark(dev_bases)
    FAULT_DIR=_FAULT_DIR_MAIN
    assert not set(dev_bases.record_id)&set(fault_bases.record_id)
    dev_feat,dev_E=two_stage_features(dev_index)
    dev_rows=[]
    for tau in TS["tau_grid"]:
        comp,_=two_stage_predict(dev_feat,dev_E,dev_pairs,tau);sc=two_stage_score(dev_pairs,comp);dev_rows.append({"tau":tau,**{k:v for k,v in sc.items() if k!="strata"}})
    dev_table=pd.DataFrame(dev_rows);dev_table.to_csv(TAB_DIR/"Two_Stage_Detector_Development.csv",index=False)
    # Deterministic selection rule: among the grid values whose unrelated-pair false-positive rate does not
    # exceed the ceiling, take those maximising the minimum stratum recall; break ties by the largest tau.
    feasible=dev_table.loc[dev_table.fpr<=CONFIG["fault_max_fpr"]]
    if feasible.empty:feasible=dev_table
    best=feasible.sort_values(["min_stratum_recall","tau"],ascending=[False,False]).iloc[0];tau_selected=float(best.tau)
    log(f"Two-stage detector development (dev set, {len(dev_bases)} bases):\n"+dev_table.round(4).to_string(index=False)+f"\nSelected tau = {tau_selected}")

    # Single evaluation on the frozen 90-base benchmark.
    main_feat,main_E=two_stage_features(fault_index)
    comp,dire=two_stage_predict(main_feat,main_E,pair_eval,tau_selected)
    score=two_stage_score(pair_eval,comp)
    truth=pair_eval.near_true.to_numpy()
    two_stage_strata=pd.DataFrame([{"fault_type":ft,"injected_n":int(((truth==1)&(pair_eval.fault_type==ft)).sum()),
        "detected":int(comp[(truth==1)&(pair_eval.fault_type==ft).to_numpy()].sum()),"same_component_recall":rec,
        "direct_pairwise_recall":float(dire[(truth==1)&(pair_eval.fault_type==ft).to_numpy()].mean())} for ft,rec in score["strata"].items()])
    two_stage_strata.to_csv(TAB_DIR/"Two_Stage_Detector_Results.csv",index=False)
    two_stage_results={"status":"completed","tau":tau_selected,"candidate_radius":TS["candidate_radius"],
        "pooled_recall":score["pooled_recall"],"min_stratum_recall":score["min_stratum_recall"],
        "false_positive_count":score["fp_count"],"n_controls":score["n_controls"],"fpr":score["fpr"],
        "direct_pairwise_recall":float(dire[truth==1].mean()),
        "passes_stratum_rule":bool(score["min_stratum_recall"]>=CONFIG["fault_min_recall"] and score["fpr"]<=CONFIG["fault_max_fpr"])}
    (RAW_DIR/"two_stage_detector.json").write_text(json.dumps(two_stage_results,indent=2),encoding="utf-8")
    log("Two-stage detector on the frozen benchmark:\n"+json.dumps(two_stage_results,indent=2)+"\n"+two_stage_strata.round(4).to_string(index=False))
    ts_encoder.cpu();del ts_encoder;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()


## 8. Cross-modality portability on PathMNIST



In [ ]:
path_cache=CACHE_DIR/"pathmnist_audit_radius4.csv"
if path_cache.exists():
    path_audit=pd.read_csv(path_cache)
    for col in [c for c in path_audit if c.endswith(("_conflict","_leakage")) or c in ("readable","metadata_valid")]:
        path_audit[col]=path_audit[col].astype(bool)
else:
    path_audit=audit_manifest(pathmnist_index,CONFIG["phash_radius"],progress_label="PathMNIST audit")
    path_audit.to_csv(path_cache,index=False)

def audit_summary(df,name,modality,file_format,provenance):
    return {"dataset":name,"modality":modality,"file_format":file_format,
        "label_classes":int(df.class_name.nunique()),"provenance_availability":provenance,
        "files":int(len(df)),"unique_sha256":int(df.sha256.nunique()),
        "exact_conflict_groups":int(df.groupby("exact_group").class_name.nunique().gt(1).sum()),
        "phash_conflict_groups":int(df.groupby("phash_group").class_name.nunique().gt(1).sum()),
        "exact_split_leakage_groups":int(df.groupby("exact_group").source_split.nunique().gt(1).sum()),
        "phash_split_leakage_groups":int(df.groupby("phash_group").source_split.nunique().gt(1).sum()),
        "root_manifest_hash":root_manifest_hash(df)}

path_summary=audit_summary(path_audit,"PathMNIST","Colorectal histopathology","PNG","split and class; no patient/study/device identifiers")
(RAW_DIR/"PathMNIST_integrity_audit.json").write_text(json.dumps(path_summary,indent=2),encoding="utf-8")

# The same metadata-control interface is executed; fixed dimensions are retained
# but standardized safely by scikit-learn. Results are not interpreted clinically.
path_meta=path_audit.copy()
train_mask=path_meta.source_split.eq("train");test_mask=path_meta.source_split.eq("test")
path_shortcut=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=500,random_state=SEED))
path_shortcut.fit(path_meta.loc[train_mask,meta_cols],path_meta.loc[train_mask,"label"])
path_pred=path_shortcut.predict(path_meta.loc[test_mask,meta_cols])
path_shortcut_metrics={"accuracy":accuracy_score(path_meta.loc[test_mask,"label"],path_pred),
    "balanced_accuracy":balanced_accuracy_score(path_meta.loc[test_mask,"label"],path_pred),
    "macro_f1":f1_score(path_meta.loc[test_mask,"label"],path_pred,average="macro")}
pd.DataFrame([path_shortcut_metrics]).to_csv(TAB_DIR/"PathMNIST_Metadata_Control.csv",index=False)

module_rows=[]
module_status={
    "Dataset identity":"completed","Image/subject identity":"completed_partial_no_subject_ids",
    "Label integrity":"completed","Provenance":"completed_partial",
    "Partition integrity":"completed","Shortcut controls":"completed",
    "Reliability evidence":"not_evaluated_no_image_model","Claim gate":"completed",
}
for module,status in module_status.items():
    module_rows.append({"dataset":"PathMNIST","module":module,"status":status,
        "core_code_changed":False,"configuration_only":True})
portability_modules=pd.DataFrame(module_rows)
portability_modules.to_csv(TAB_DIR/"PathMNIST_Module_Completion.csv",index=False)
completed=int(portability_modules.status.str.startswith("completed").sum())
portability_table=pd.DataFrame([{
    "dataset":"PathMNIST","modality":"Colorectal histopathology","file_format":"PNG",
    "label_structure":f"{path_audit.class_name.nunique()}-class single-label",
    "provenance_availability":"Partial; no patient/study/device identifiers",
    "modules_completed":completed,"total_modules":len(portability_modules),
    "adapter_effort":"Configuration and export adapter only; core audit unchanged",
    "schema_valid":"pending passport assembly","unresolved_evidence":"Patient/study identity and model reliability",
}])
portability_table.to_csv(TAB_DIR/"Table5_Cross_Dataset_Portability.csv",index=False)
log("PathMNIST portability evidence:\n"+json.dumps(path_summary,indent=2))



## 9. Reference CXR model, calibration, conformal, referral, and shortcut evidence

This cell is adapted from the supplied CXR notebook. It regenerates the
inherited model evidence only when `run_reference_models=True`. It is not used
as the new framework's controlled ground truth.



In [ ]:
reference_results={"status":"not_evaluated"}
if CONFIG["run_reference_models"]:
    normalization_mean=[0.485,0.456,0.406];normalization_std=[0.229,0.224,0.225]
    transform=transforms.Compose([transforms.Resize((CONFIG["image_size"],CONFIG["image_size"]),interpolation=InterpolationMode.BILINEAR),
        transforms.ToTensor(),transforms.Normalize(normalization_mean,normalization_std)])
    class CXRDataset(Dataset):
        def __init__(self,frame):self.paths=frame.path.tolist();self.labels=frame.label.to_numpy(np.int64)
        def __len__(self):return len(self.paths)
        def __getitem__(self,i):
            with Image.open(self.paths[i]) as image:x=transform(image.convert("RGB"))
            return x,int(self.labels[i])
    def loader(frame):
        return DataLoader(CXRDataset(frame),batch_size=CONFIG["batch_size"],shuffle=False,
            num_workers=CONFIG["num_workers"],pin_memory=DEVICE.type=="cuda")
    embedding_file=CACHE_DIR/"CXR_frozen_embeddings.npz"
    if embedding_file.exists():
        cache=np.load(embedding_file);emb={n:cache[f"x_{n}"] for n in cxr_frames};lab={n:cache[f"y_{n}"] for n in cxr_frames}
    else:
        encoder=timm.create_model(CONFIG["encoder"],pretrained=True,num_classes=0,global_pool="avg").to(DEVICE).eval()
        for p in encoder.parameters():p.requires_grad=False
        emb={};lab={}
        with torch.inference_mode():
            for name,frame in cxr_frames.items():
                xb=[];yb=[]
                for j,(images,target) in enumerate(loader(frame),1):
                    with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP_ENABLED):features=encoder(images.to(DEVICE,non_blocking=True))
                    xb.append(features.float().cpu().numpy());yb.append(target.numpy())
                    if j%20==0:log(f"embeddings {name}: {min(j*CONFIG['batch_size'],len(frame)):,}/{len(frame):,}")
                emb[name]=np.concatenate(xb);lab[name]=np.concatenate(yb)
        np.savez_compressed(embedding_file,**{f"x_{n}":v for n,v in emb.items()},**{f"y_{n}":v for n,v in lab.items()})
        encoder.cpu();del encoder;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    X_train,X_val,X_cal,X_test=[emb[n] for n in ("train","validation","calibration","test")]
    y_train,y_val,y_cal,y_test=[lab[n] for n in ("train","validation","calibration","test")]
    direct_candidates={}
    for c in (.1,1.,10.):
        model=make_pipeline(StandardScaler(),LogisticRegression(C=c,class_weight="balanced",max_iter=500,random_state=SEED))
        model.fit(X_train,y_train);direct_candidates[f"Logistic_C{c}"]=(model,f1_score(y_val,model.predict(X_val),average="macro"))
    best_name=max(direct_candidates,key=lambda n:direct_candidates[n][1]);direct_model=direct_candidates[best_name][0]
    abnormal_model=make_pipeline(StandardScaler(),LogisticRegression(C=1,class_weight="balanced",max_iter=500,random_state=SEED))
    abnormal_model.fit(X_train,(y_train!=0).astype(int))
    disease_mask=y_train!=0;disease_model=make_pipeline(StandardScaler(),LogisticRegression(C=1,class_weight="balanced",max_iter=500,random_state=SEED))
    disease_model.fit(X_train[disease_mask],(y_train[disease_mask]==2).astype(int))

    def logits_for(model,x):
        scores=model.decision_function(x)
        return np.column_stack([-scores/2,scores/2]) if scores.ndim==1 else scores
    def fit_temperature(logits,target):
        result=minimize_scalar(lambda lt:log_loss(target,softmax(logits/np.exp(lt),axis=1),labels=np.arange(logits.shape[1])),bounds=(-3,3),method="bounded")
        return float(np.exp(result.x))
    td=fit_temperature(logits_for(direct_model,X_val),y_val)
    ta=fit_temperature(logits_for(abnormal_model,X_val),(y_val!=0).astype(int))
    vd=y_val!=0;tt=fit_temperature(logits_for(disease_model,X_val[vd]),(y_val[vd]==2).astype(int))
    def calibrated(model,x,t):return softmax(logits_for(model,x)/t,axis=1)
    def hierarchical(x):
        pa=calibrated(abnormal_model,x,ta)[:,1];conditional=calibrated(disease_model,x,tt)
        return np.column_stack([1-pa,pa*conditional[:,0],pa*conditional[:,1]])
    direct_test=calibrated(direct_model,X_test,td);hier_cal=hierarchical(X_cal);hier_test=hierarchical(X_test)

    def brier(y,p):return float(np.mean(np.sum((p-np.eye(p.shape[1])[y])**2,axis=1)))
    def ece(y,p,bins=15):
        pred=p.argmax(1);conf=p.max(1);correct=pred==y;edges=np.linspace(0,1,bins+1);value=0.
        for i in range(bins):
            mask=(conf>=edges[i])&(conf<(edges[i+1] if i<bins-1 else edges[i+1]+1e-12))
            if mask.any():value+=mask.mean()*abs(correct[mask].mean()-conf[mask].mean())
        return float(value)
    def model_metrics(y,p):
        pred=p.argmax(1)
        return {"accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),
            "macro_f1":f1_score(y,pred,average="macro"),"macro_auc_ovr":roc_auc_score(y,p,multi_class="ovr",average="macro"),
            "ece":ece(y,p),"brier":brier(y,p),"nll":log_loss(y,p,labels=np.arange(3))}
    # v0.3: canonical digest of each model's test-set probability matrix, so a gate can be tied to the exact
    # predictions it was computed from rather than to the procedure that produces them.
    def prediction_digest(p):return sha256_bytes(np.ascontiguousarray(np.round(np.asarray(p,dtype=np.float64),12)).tobytes())
    probs={"direct":direct_test,"hierarchical":hier_test}
    test_metrics=pd.DataFrame([{"model":name,**model_metrics(y_test,p)} for name,p in probs.items()])
    test_metrics.to_csv(TAB_DIR/"CXR_Test_Metrics.csv",index=False)

    bootstrap=[];class_ids=[np.flatnonzero(y_test==k) for k in range(3)]
    for rep in range(CONFIG["bootstrap_replicates"]):
        rng=np.random.default_rng(SEED+rep);ids=np.concatenate([rng.choice(g,len(g),replace=True) for g in class_ids])
        for name,p in probs.items():bootstrap.append({"replicate":rep,"model":name,**model_metrics(y_test[ids],p[ids])})
    boot=pd.DataFrame(bootstrap);boot.to_csv(TAB_DIR/"CXR_Bootstrap_Samples.csv",index=False)
    ci=[]
    for name,group in boot.groupby("model"):
        for metric in ("accuracy","balanced_accuracy","macro_f1","macro_auc_ovr","ece","brier","nll"):
            ci.append({"model":name,"metric":metric,"ci_low":group[metric].quantile(.025),"ci_high":group[metric].quantile(.975)})
    pd.DataFrame(ci).to_csv(TAB_DIR/"CXR_Bootstrap_95CI.csv",index=False)

    def higher_quantile(values,alpha):
        level=min(1.,np.ceil((len(values)+1)*(1-alpha))/len(values));return float(np.quantile(values,level,method="higher"))
    q={k:higher_quantile(1-hier_cal[y_cal==k,k],CONFIG["conformal_alpha"]) for k in range(3)}
    # v0.2: per-class calibration sizes and LAC thresholds (score 1 - p_hat_y; quantile ceil((n_k+1)(1-alpha))/n_k, "higher").
    pd.DataFrame([{"class":CLASS_NAMES[k],"n_calibration":int((y_cal==k).sum()),"q_k":q[k],
                   "quantile_level":min(1.,np.ceil((int((y_cal==k).sum())+1)*(1-CONFIG["conformal_alpha"]))/int((y_cal==k).sum()))} for k in range(3)]
                ).to_csv(TAB_DIR/"CXR_Conformal_Class_Thresholds.csv",index=False)
    sets=[[k for k in range(3) if 1-row[k]<=q[k]] for row in hier_test]
    covered=np.array([target in s for target,s in zip(y_test,sets)]);sizes=np.array([len(s) for s in sets])
    conformal={"marginal_coverage":covered.mean(),"mean_set_size":sizes.mean(),"singleton_rate":np.mean(sizes==1),
        "empty_sets":int(np.sum(sizes==0)),**{f"{CLASS_NAMES[k]}_coverage":covered[y_test==k].mean() for k in range(3)}}
    pd.DataFrame([conformal]).to_csv(TAB_DIR/"CXR_Conformal_Summary.csv",index=False)
    def sensitivity_threshold(y,score,target):
        _,tpr,thresholds=roc_curve(y,score);feasible=np.flatnonzero(tpr>=target);return float(thresholds[feasible[0]])
    abnormal_threshold=sensitivity_threshold((y_cal!=0).astype(int),1-hier_cal[:,0],CONFIG["target_abnormal_sensitivity"])
    mask=y_cal!=0;cond_cal=hier_cal[mask,1:];cond_cal/=cond_cal.sum(1,keepdims=True)
    tb_threshold=sensitivity_threshold((y_cal[mask]==2).astype(int),cond_cal[:,1],CONFIG["target_tb_sensitivity"])
    def entropy(p):p=np.clip(p,1e-8,1);return -np.sum(p*np.log(p),axis=1)/np.log(p.shape[1])
    entropy_threshold=float(np.quantile(entropy(hier_cal),CONFIG["target_auto_coverage"]))
    cond=hier_test[:,1:]/hier_test[:,1:].sum(1,keepdims=True);op=np.zeros(len(hier_test),dtype=int)
    abnormal=(1-hier_test[:,0])>=abnormal_threshold;op[abnormal]=np.where(cond[abnormal,1]>=tb_threshold,2,1)
    singleton=sizes==1;single_label=np.array([s[0] if len(s)==1 else -1 for s in sets])
    accepted=singleton&(single_label==op)&(entropy(hier_test)<=entropy_threshold)
    selective={"coverage":accepted.mean(),"accepted_accuracy":accuracy_score(y_test[accepted],op[accepted]) if accepted.any() else np.nan,
        "selective_risk":1-accuracy_score(y_test[accepted],op[accepted]) if accepted.any() else np.nan,
        "referred_n":int((~accepted).sum()),
        "accepted_n":int(accepted.sum()),
        "selective_risk_rule_of_three_upper":float(3/max(int(accepted.sum()),1))}   # v0.2
    pd.DataFrame([selective]).to_csv(TAB_DIR/"CXR_Selective_Triage.csv",index=False)

    shortcut_model=make_pipeline(StandardScaler(),LogisticRegression(class_weight="balanced",max_iter=500,random_state=SEED))
    shortcut_model.fit(train_df[meta_cols],train_df.label);shortcut_pred=shortcut_model.predict(test_df[meta_cols])
    shortcut={"accuracy":accuracy_score(y_test,shortcut_pred),"balanced_accuracy":balanced_accuracy_score(y_test,shortcut_pred),
        "macro_f1":f1_score(y_test,shortcut_pred,average="macro")}
    pd.DataFrame([shortcut]).to_csv(TAB_DIR/"CXR_Metadata_Shortcut_Diagnostic.csv",index=False)
    reference_results={"status":"completed","selected_model":best_name,
        "prediction_digests":{name:prediction_digest(p) for name,p in probs.items()},"temperatures":{"direct":td,"abnormal":ta,"disease":tt},
        "test_metrics":test_metrics.set_index("model").to_dict("index"),"conformal":conformal,"selective":selective,"metadata_shortcut":shortcut}
    (RAW_DIR/"CXR_reference_model_evidence.json").write_text(json.dumps(reference_results,indent=2),encoding="utf-8")
    log("Reference model evidence:\n"+json.dumps(reference_results,indent=2))



## 9b. End-to-end fine-tuned CNN on the same locked partitions (v0.2)

Same MobileNetV3-Small architecture initialised from ImageNet weights, all layers trainable, same seed and partitions; temperature scaling, conformal sets, selective thresholds, and the paired bootstrap are then applied exactly as for the frozen-feature models. Gate C4 is re-evaluated against the stronger of the frozen-feature direct model and the fine-tuned model.


In [ ]:
if CONFIG["run_reference_models"] and CONFIG["run_finetuned_model"]:
    FT=CONFIG["finetune"];torch.manual_seed(SEED)
    train_transform=transforms.Compose([transforms.RandomResizedCrop(CONFIG["image_size"],scale=tuple(FT["crop_scale"]),interpolation=InterpolationMode.BILINEAR)]
        +([transforms.RandomHorizontalFlip()] if FT["hflip"] else [])+[transforms.ToTensor(),transforms.Normalize(normalization_mean,normalization_std)])
    class CXRDatasetAug(CXRDataset):
        def __getitem__(self,i):
            with Image.open(self.paths[i]) as image:x=train_transform(image.convert("RGB"))
            return x,int(self.labels[i])
    g=torch.Generator();g.manual_seed(SEED)
    train_loader=DataLoader(CXRDatasetAug(train_df),batch_size=FT["batch_size"],shuffle=True,num_workers=CONFIG["num_workers"],pin_memory=DEVICE.type=="cuda",generator=g)
    ft_model=timm.create_model(CONFIG["encoder"],pretrained=True,num_classes=3).to(DEVICE)
    counts=np.bincount(train_df.label.to_numpy(),minlength=3);class_weights=torch.tensor(len(train_df)/(3*counts),dtype=torch.float32).to(DEVICE)
    criterion=torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer=torch.optim.AdamW(ft_model.parameters(),lr=FT["lr"],weight_decay=FT["weight_decay"])
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=FT["max_epochs"])
    scaler=torch.amp.GradScaler(enabled=AMP_ENABLED)

    def ft_logits(frame):
        ft_model.eval();out=[]
        with torch.inference_mode():
            for images,_ in loader(frame):
                with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP_ENABLED):out.append(ft_model(images.to(DEVICE,non_blocking=True)).float().cpu().numpy())
        return np.concatenate(out)

    best_f1=-1.;patience=0;history=[];ft_weights=MODEL_DIR/"finetuned_best.pt"
    for epoch in range(1,FT["max_epochs"]+1):
        ft_model.train();running=0.;seen=0
        for images,targets in train_loader:
            images=images.to(DEVICE,non_blocking=True);targets=targets.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type,enabled=AMP_ENABLED):loss=criterion(ft_model(images),targets)
            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
            running+=float(loss)*len(targets);seen+=len(targets)
        scheduler.step()
        val_f1=f1_score(y_val,ft_logits(val_df).argmax(1),average="macro")
        history.append({"epoch":epoch,"train_loss":running/max(seen,1),"val_macro_f1":val_f1})
        log(f"fine-tune epoch {epoch}: loss {running/max(seen,1):.4f}, validation macro-F1 {val_f1:.4f}")
        if val_f1>best_f1:best_f1=val_f1;patience=0;torch.save(ft_model.state_dict(),ft_weights);best_epoch=epoch
        else:
            patience+=1
            if patience>=FT["patience"]:break
    pd.DataFrame(history).to_csv(TAB_DIR/"CXR_FineTuned_Training_History.csv",index=False)
    ft_model.load_state_dict(torch.load(ft_weights,map_location=DEVICE))

    ft_val_logits=ft_logits(val_df);ft_cal_logits=ft_logits(calibration_df);ft_test_logits=ft_logits(test_df)
    t_ft=fit_temperature(ft_val_logits,y_val)
    ft_cal=softmax(ft_cal_logits/t_ft,axis=1);ft_test=softmax(ft_test_logits/t_ft,axis=1)
    ft_metrics=model_metrics(y_test,ft_test)
    pd.DataFrame([{"model":"finetuned",**ft_metrics}]).to_csv(TAB_DIR/"CXR_FineTuned_Test_Metrics.csv",index=False)

    # Same class-stratified bootstrap resamples as the frozen-feature models (identical seeds -> paired).
    ft_boot=[]
    for rep in range(CONFIG["bootstrap_replicates"]):
        rng=np.random.default_rng(SEED+rep);ids=np.concatenate([rng.choice(gi,len(gi),replace=True) for gi in class_ids])
        ft_boot.append({"replicate":rep,"model":"finetuned",**model_metrics(y_test[ids],ft_test[ids])})
    ft_boot=pd.DataFrame(ft_boot);ft_boot.to_csv(TAB_DIR/"CXR_FineTuned_Bootstrap_Samples.csv",index=False)
    ft_ci=[{"model":"finetuned","metric":m,"ci_low":ft_boot[m].quantile(.025),"ci_high":ft_boot[m].quantile(.975)} for m in ("accuracy","balanced_accuracy","macro_f1","macro_auc_ovr","ece","brier","nll")]
    pd.DataFrame(ft_ci).to_csv(TAB_DIR/"CXR_FineTuned_Bootstrap_95CI.csv",index=False)
    paired=ft_boot.set_index("replicate").macro_f1-boot.loc[boot.model.eq("direct")].set_index("replicate").macro_f1
    ft_contrast={"metric":"macro_f1","mean_finetuned_minus_direct":float(paired.mean()),"ci_low":float(paired.quantile(.025)),"ci_high":float(paired.quantile(.975))}

    # Same conformal and selective procedure on the calibration partition.
    q_ft={k:higher_quantile(1-ft_cal[y_cal==k,k],CONFIG["conformal_alpha"]) for k in range(3)}
    sets_ft=[[k for k in range(3) if 1-row[k]<=q_ft[k]] for row in ft_test]
    covered_ft=np.array([t in s for t,s in zip(y_test,sets_ft)]);sizes_ft=np.array([len(s) for s in sets_ft])
    conformal_ft={"marginal_coverage":float(covered_ft.mean()),"mean_set_size":float(sizes_ft.mean()),"singleton_rate":float(np.mean(sizes_ft==1)),
        "empty_sets":int(np.sum(sizes_ft==0)),**{f"{CLASS_NAMES[k]}_coverage":float(covered_ft[y_test==k].mean()) for k in range(3)},
        "q":{CLASS_NAMES[k]:q_ft[k] for k in range(3)}}
    pd.DataFrame([{k:v for k,v in conformal_ft.items() if k!="q"}]).to_csv(TAB_DIR/"CXR_FineTuned_Conformal_Summary.csv",index=False)

    reference_results["finetuned"]={"prediction_digest":prediction_digest(ft_test),"epochs_trained":len(history),"best_epoch":int(best_epoch),"temperature":t_ft,"test_metrics":ft_metrics,
        "bootstrap_95ci":{r["metric"]:[r["ci_low"],r["ci_high"]] for r in ft_ci},"paired_contrast_vs_direct":ft_contrast,"conformal":conformal_ft,
        "hyperparameters":FT}
    (RAW_DIR/"CXR_finetuned_model_evidence.json").write_text(json.dumps(reference_results["finetuned"],indent=2),encoding="utf-8")
    log("Fine-tuned model evidence:\n"+json.dumps(reference_results["finetuned"],indent=2))
    ft_model.cpu();del ft_model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()


## 10. Repeated-run determinism, scaling, and optional second-environment comparison



In [ ]:
tracemalloc.start();repeat_start=time.perf_counter()
repeat_a=audit_manifest(fault_index,CONFIG["phash_radius"],progress_label="repeat A")
repeat_a_seconds=time.perf_counter()-repeat_start;_,repeat_a_peak=tracemalloc.get_traced_memory();tracemalloc.stop()
tracemalloc.start();repeat_start=time.perf_counter()
repeat_b=audit_manifest(fault_index,CONFIG["phash_radius"],progress_label="repeat B")
repeat_b_seconds=time.perf_counter()-repeat_start;_,repeat_b_peak=tracemalloc.get_traced_memory();tracemalloc.stop()
digest_columns=["record_id","sha256","pixel_sha256","phash","exact_group","pixel_group","phash_group",
                "exact_label_conflict","exact_split_leakage","metadata_valid"]
digest_a=canonical_digest(repeat_a,digest_columns);digest_b=canonical_digest(repeat_b,digest_columns)
ari=adjusted_rand_score(repeat_a.sort_values("record_id").phash_group,repeat_b.sort_values("record_id").phash_group)

scaling_rows=[]
for size in CONFIG["scaling_sizes"]:
    sample=pathmnist_index.sample(min(size,len(pathmnist_index)),random_state=SEED).reset_index(drop=True)
    tracemalloc.start();start=time.perf_counter();sample_audit=audit_manifest(sample,CONFIG["phash_radius"],workers=4,progress_label=f"scaling n={len(sample)}")
    seconds=time.perf_counter()-start;_,peak=tracemalloc.get_traced_memory();tracemalloc.stop()
    scaling_rows.append({"n_images":len(sample),"runtime_seconds":seconds,"peak_memory_mb":peak/2**20,
        "images_per_second":len(sample)/max(seconds,1e-9),"n_phash_components":sample_audit.phash_group.nunique()})
scaling=pd.DataFrame(scaling_rows);scaling.to_csv(TAB_DIR/"Computational_Scaling.csv",index=False)

current_bundle={"environment":environment,"exact_digest":digest_a,"repeat_digest":digest_b,
    "within_environment_exact_agreement":float(digest_a==digest_b),"within_environment_phash_ari":float(ari),
    "threshold_metrics":threshold_sensitivity[["radius","precision","recall","f1","false_positive_rate"]].to_dict("records")}
bundle_path=COMPARE_DIR/f"reproducibility_{environment['fingerprint'][:12]}.json"
bundle_path.write_text(json.dumps(current_bundle,indent=2),encoding="utf-8")
other_bundles=[]
for p in COMPARE_DIR.glob("reproducibility_*.json"):
    data=json.loads(p.read_text())
    if data.get("environment",{}).get("fingerprint")!=environment["fingerprint"]:other_bundles.append((p,data))
if other_bundles:
    other_path,other=other_bundles[0]
    cross_status="completed"
    cross_exact=float(other.get("exact_digest")==digest_a)
    cross_note=f"Compared with {other_path.name}"
else:
    cross_status="not_evaluated"
    cross_exact=np.nan
    cross_note="Copy this output folder to a second Colab/runtime and rerun to enable comparison."

reproducibility_table=pd.DataFrame([
    {"environment":"Primary repeated","configuration":"Frozen","exact_output_agreement":float(digest_a==digest_b),
     "approximate_group_agreement":ari,"runtime_seconds":np.mean([repeat_a_seconds,repeat_b_seconds]),"peak_memory_mb":max(repeat_a_peak,repeat_b_peak)/2**20,"status":"completed"},
    {"environment":"Secondary","configuration":"Frozen","exact_output_agreement":cross_exact,
     "approximate_group_agreement":np.nan,"runtime_seconds":np.nan,"peak_memory_mb":np.nan,"status":cross_status},
    {"environment":"Threshold sensitivity","configuration":str(CONFIG["phash_radii"]),"exact_output_agreement":np.nan,
     "approximate_group_agreement":threshold_sensitivity.f1.mean(),"runtime_seconds":np.nan,"peak_memory_mb":np.nan,"status":"completed"},
])
reproducibility_table.to_csv(TAB_DIR/"Table4_Reproducibility_Sensitivity.csv",index=False)
log("Reproducibility:\n"+reproducibility_table.to_string(index=False)+"\n"+cross_note)



## 11. Versioned passport schema and transparent claim gates



In [ ]:
PASSPORT_SCHEMA={
    "$schema":"https://json-schema.org/draft/2020-12/schema","$id":"https://example.org/imaging-passport.schema-1.1.1.json",
    "title":"Imaging Dataset Reliability Passport","type":"object",
    "required":["schema_version","passport_id","created_at","dataset_identity","execution_identity","modules","claim_gates"],
    "properties":{
        "schema_version":{"type":"string"},"passport_id":{"type":"string"},"created_at":{"type":"string"},
        "dataset_identity":{"type":"object","required":["name","root_manifest_hash","file_count"],
            "properties":{"name":{"type":"string"},"version":{"type":["string","null"]},"access_date":{"type":"string"},
                "root_manifest_hash":{"type":"string"},"file_count":{"type":"integer"},"license":{"type":["string","null"]}}},
        "execution_identity":{"type":"object","required":["software_version","configuration_hash","environment_fingerprint"],
            "properties":{"software_version":{"type":"string"},"configuration_hash":{"type":"string"},"environment_fingerprint":{"type":"string"}}},
        "modules":{"type":"object","required":["image_identity","label_integrity","provenance","partition_integrity","shortcut_controls","reliability_evidence"],
            "additionalProperties":{"type":["object","array","string","number","boolean","null"]}},
        "claim_gates":{"type":"array","items":{"type":"object",
            "required":["claim_id","claim","state","rule_id","rationale","depends_on","effective_state","threshold_provenance","model_scope"],   # v0.3: schema 1.1.1
            "properties":{"claim_id":{"type":"string"},"claim":{"type":"string"},
                "state":{"enum":["supported_in_evaluated_setting","conditional","blocked","not_evaluated"]},
                "rule_id":{"type":"string"},"evidence":{"type":["object","array","string","number","null"]},"rationale":{"type":"string"},
                "depends_on":{"type":"array","items":{"type":"string"}},
                "effective_state":{"enum":["supported_in_evaluated_setting","conditional","blocked","not_evaluated"]},
                "model_scope":{"type":["object","null"],"required":["model","determined_by","prediction_digest"],   # v0.3
                    "description":"The model a model-dependent gate was computed from; null for gates that do not depend on a fitted model.",
                    "properties":{"model":{"type":"string"},"determined_by":{"type":"object"},
                        "prediction_digest":{"type":["string","null"]}}},
                "threshold_provenance":{"type":["object","null"],"required":["config_key","value","rationale","override"],
                    "properties":{"config_key":{"type":["string","null"]},"value":{"type":["number","null"]},"rationale":{"type":"string"},
                        "override":{"type":["object","null"],"required":["value","by","rationale","timestamp"],
                            "properties":{"value":{"type":"number"},"by":{"type":"string"},"rationale":{"type":"string"},"timestamp":{"type":"string"}}}}}}}}
    }
}
(RAW_DIR/"passport.schema.json").write_text(json.dumps(PASSPORT_SCHEMA,indent=2),encoding="utf-8")

# v0.2: dependency relation and threshold provenance on every gate record.
THRESHOLD_KEYS={"C1":("fault_min_recall","applied to the minimum recall over injected transformation strata (stratum-minimum rule)"),
                "C4":("shortcut_gap_tolerance","software-governance default"),"C5":("conformal_tolerance","software-governance default")}
RULE_NOTES={"C2":"rule fires on any asserted identity group crossing partitions; no numeric threshold",
            "C3":"requires patient and institution identifiers; no numeric threshold",
            "C6":"supported when at least six of eight modules complete without core-code change",
            "C7":"requires exact-output agreement of 1.0 across environments"}
def claim_gate(claim_id,claim,state,rule_id,evidence,rationale,model_scope=None):
    key,note=THRESHOLD_KEYS.get(claim_id,(None,RULE_NOTES.get(claim_id,"")))
    return {"claim_id":claim_id,"claim":claim,"state":state,"rule_id":rule_id,"evidence":evidence,"rationale":rationale,
            "depends_on":list(CONFIG["gate_dependencies"][claim_id]),
            "threshold_provenance":{"config_key":key,"value":(CONFIG[key] if key else None),"rationale":note,"override":None},
            "model_scope":model_scope}

# v0.3: a reliability gate is ambiguous unless the model it was computed from is named. C4 is scoped to the
# best-performing image model (a shortcut claim must be tested against the strongest available representation);
# C5 to the hierarchical model, on whose probabilities the conformal sets and the selective policy were built.
def model_scope(name,determined_by,digest):
    return {"model":name,"determined_by":determined_by,"prediction_digest":digest}
DETERMINISTIC={"software_version":CONFIG["software_version"],"seed":SEED,"encoder":CONFIG["encoder"],
               "note":"Frozen-feature models are deterministic given the configuration hash and seed."}

def propagate_dependencies(gates):
    by_id={g["claim_id"]:g for g in gates}
    def effective(cid):
        g=by_id[cid];pre=[effective(d) for d in g["depends_on"]]
        if g["state"]=="supported_in_evaluated_setting" and any(p!="supported_in_evaluated_setting" for p in pre):return "conditional"
        return g["state"]
    for g in gates:
        g["effective_state"]=effective(g["claim_id"])
        if g["effective_state"]!=g["state"]:
            blocked=[d for d in g["depends_on"] if by_id[d]["effective_state" if "effective_state" in by_id[d] else "state"]!="supported_in_evaluated_setting"]
            g["rationale"]+=f" Prerequisite {', '.join(blocked)} not supported; effective state downgraded to conditional."
    return gates

# v0.2: C1 is keyed on the weakest injected stratum, not on the pooled design-weighted recall.
deterministic_ok=bool((fault_metrics.loc[fault_metrics.fault_type!="Near duplicate","recall"]>=CONFIG["fault_min_recall"]).all())
fault_pass=bool(deterministic_ok and min_stratum_recall>=CONFIG["fault_min_recall"] and (fault_metrics.false_positive_rate<=CONFIG["fault_max_fpr"]).all())
gates=[]
gates.append(claim_gate("C1","Controlled faults are detected at prespecified performance thresholds",
    "supported_in_evaluated_setting" if fault_pass else "blocked","R-FAULT-001",
    {"min_stratum_recall":min_stratum_recall,"min_stratum":min_stratum,"pooled_recall_design_weighted":float(fault_metrics.loc[fault_metrics.fault_type=="Near duplicate","recall"].iloc[0]),
     "design_based_se":design_se,"max_fpr":float(fault_metrics.false_positive_rate.max()),
     "two_stage_detector":{k:two_stage_results.get(k) for k in ("status","tau","min_stratum_recall","fpr","passes_stratum_rule")}},
    "All detectors meet both thresholds on every stratum." if fault_pass else f"Stratum {min_stratum} recall {min_stratum_recall:.3f} is below the {CONFIG['fault_min_recall']} floor."))
gates.append(claim_gate("C2","The reconstructed CXR test split is independent at detected file/image-identity levels",
    "supported_in_evaluated_setting","R-LEAK-001",{"asserted_cross_partition_groups":0},
    "No exact or declared perceptual group crosses the reconstructed partitions."))
gates.append(claim_gate("C3","The CXR evaluation is patient- and institution-independent","not_evaluated","R-PROV-001",None,
    "Patient and institution identifiers are unavailable."))

if reference_results.get("status")=="completed":
    direct_f1=float(reference_results["test_metrics"]["direct"]["macro_f1"])
    metadata_f1=float(reference_results["metadata_shortcut"]["macro_f1"])
    finetuned_f1=float(reference_results["finetuned"]["test_metrics"]["macro_f1"]) if "finetuned" in reference_results else None
    best_image_f1=max([direct_f1]+([finetuned_f1] if finetuned_f1 is not None else []))   # v0.2
    shortcut_block=metadata_f1>=best_image_f1-CONFIG["shortcut_gap_tolerance"]
    digests=reference_results.get("prediction_digests",{})
    best_is_ft=finetuned_f1 is not None and finetuned_f1>=direct_f1
    c4_scope=model_scope(("best image model: fine-tuned %s, all layers trainable"%CONFIG["encoder"]) if best_is_ft
                         else ("best image model: direct %s on frozen %s features"%(best_name,CONFIG["encoder"])),
                         dict(DETERMINISTIC,**({"hyperparameters":CONFIG["finetune"],
                                                "best_epoch":reference_results["finetuned"]["best_epoch"],
                                                "temperature":reference_results["finetuned"]["temperature"],
                                                "checkpoint":str(MODEL_DIR/"finetuned_best.pt")} if best_is_ft else {})),
                         (reference_results["finetuned"]["prediction_digest"] if best_is_ft else digests.get("direct")))
    gates.append(claim_gate("C4","Internal CXR image performance supports clinical generalization",
        "blocked" if shortcut_block else "conditional","R-SHORTCUT-001",
        {"image_macro_f1":direct_f1,"finetuned_macro_f1":finetuned_f1,"best_image_macro_f1":best_image_f1,"metadata_macro_f1":metadata_f1,"gap":best_image_f1-metadata_f1},
        "Metadata-only performance approaches image-model performance." if shortcut_block else "External source-aware validation remains required.",
        model_scope=c4_scope))
    coverage=float(reference_results["conformal"]["marginal_coverage"]);target=1-CONFIG["conformal_alpha"]
    cov_ok=coverage>=target-CONFIG["conformal_tolerance"]
    gates.append(claim_gate("C5","Conformal marginal coverage meets the internal tolerance (hierarchical model)",
        "supported_in_evaluated_setting" if cov_ok else "blocked","R-CONFORMAL-001",
        {"observed":coverage,"target":target,"tolerance":CONFIG["conformal_tolerance"]},
        "Coverage is evaluated only within the reconstructed CXR setting, and only for the hierarchical model, "
        "on which the conformal sets and the selective policy were built.",
        model_scope=model_scope("hierarchical (temperature-scaled two-stage logistic on frozen %s features)"%CONFIG["encoder"],
                                dict(DETERMINISTIC,temperatures=reference_results["temperatures"]),digests.get("hierarchical"))))
    # v0.3: coverage of any further model is recorded as its own scoped gate, never merged into C5.
    if "finetuned" in reference_results:
        ft_cov=float(reference_results["finetuned"]["conformal"]["marginal_coverage"])
        gates.append(claim_gate("C5b","Conformal marginal coverage meets the internal tolerance (fine-tuned model)",
            "supported_in_evaluated_setting" if ft_cov>=target-CONFIG["conformal_tolerance"] else "blocked","R-CONFORMAL-001",
            {"observed":ft_cov,"target":target,"tolerance":CONFIG["conformal_tolerance"],
             "empty_sets":reference_results["finetuned"]["conformal"]["empty_sets"]},
            "Coverage of the fine-tuned model, recorded separately from C5 so that no gate mixes model scopes. "
            "The selective policy was not refitted for this model.",
            model_scope=model_scope("fine-tuned %s, all layers trainable"%CONFIG["encoder"],
                                    dict(DETERMINISTIC,hyperparameters=CONFIG["finetune"],
                                         best_epoch=reference_results["finetuned"]["best_epoch"],
                                         temperature=reference_results["finetuned"]["temperature"],
                                         checkpoint=str(MODEL_DIR/"finetuned_best.pt")),
                                    reference_results["finetuned"]["prediction_digest"])))
else:
    gates.extend([claim_gate("C4","Internal CXR image performance supports clinical generalization","not_evaluated","R-SHORTCUT-001",None,"Reference model analysis was disabled."),
                  claim_gate("C5","Conformal marginal coverage meets the internal tolerance (hierarchical model)","not_evaluated","R-CONFORMAL-001",None,"Reference model analysis was disabled.")])
gates.append(claim_gate("C6","The core passport engine transfers to a different imaging modality",
    "supported_in_evaluated_setting" if completed>=6 else "conditional","R-PORT-001",
    {"dataset":"PathMNIST","modules_completed":completed,"total_modules":len(portability_modules)},
    "Core audits ran without modification; patient identity and model reliability remain unavailable."))
gates.append(claim_gate("C7","Exact outputs reproduce across independent software environments",
    "supported_in_evaluated_setting" if cross_status=="completed" and cross_exact==1 else "not_evaluated","R-REPRO-002",
    {"cross_environment_status":cross_status,"exact_agreement":None if np.isnan(cross_exact) else cross_exact},cross_note))
gates=propagate_dependencies(gates)   # v0.2: effective states after prerequisite propagation

config_hash=sha256_bytes(json.dumps(CONFIG,sort_keys=True).encode())
passport={"schema_version":CONFIG["schema_version"],"passport_id":"IMU-CXR-PASSPORT-2026-001",
    "created_at":pd.Timestamp.utcnow().isoformat(),
    "dataset_identity":{"name":"Three-class public CXR compilation","version":None,
        "access_date":str(pd.Timestamp.utcnow().date()),"root_manifest_hash":cxr_integrity["root_manifest_hash"],
        "file_count":cxr_integrity["source_files"],"license":None},
    "execution_identity":{"software_version":CONFIG["software_version"],"configuration_hash":config_hash,
        "environment_fingerprint":environment["fingerprint"]},
    "modules":{"image_identity":cxr_integrity,"label_integrity":{"exact_conflict_files":cxr_integrity["exact_conflict_files"],
        "near_conflict_images":cxr_integrity["near_conflict_images"]},
        "provenance":{"status":"partial","missing":["patient_id","study_id","institution","device","projection"]},
        "partition_integrity":{"detected_cross_partition_groups":0,"patient_level":"not_evaluated"},
        "shortcut_controls":reference_results.get("metadata_shortcut",{"status":"not_evaluated"}),
        "reliability_evidence":reference_results},"claim_gates":gates}
jsonschema.validate(passport,PASSPORT_SCHEMA)
(RAW_DIR/"passport.json").write_text(json.dumps(passport,indent=2),encoding="utf-8")
pd.DataFrame(gates).to_json(TAB_DIR/"claim_gates.json",orient="records",indent=2)
pd.DataFrame([{k:(json.dumps(v) if k in ("evidence","threshold_provenance","model_scope") else (";".join(v) if k=="depends_on" else v)) for k,v in gate.items()} for gate in gates]).to_csv(TAB_DIR/"claim_gates.csv",index=False)

def count_schema_fields(node):
    total=len(node.get("properties",{}))
    return total+sum(count_schema_fields(v) for v in node.get("properties",{}).values() if isinstance(v,dict))
schema_summary={"schema_version":CONFIG["schema_version"],"top_level_modules":8,"declared_fields":count_schema_fields(PASSPORT_SCHEMA),
    "required_top_level_fields":len(PASSPORT_SCHEMA["required"]),"schema_valid":True,"claim_gate_count":len(gates)}
pd.DataFrame([schema_summary]).to_csv(TAB_DIR/"Passport_Schema_Summary.csv",index=False)

passport_modules=pd.DataFrame([
    {"module":"Dataset identity","status":"completed","evidence":"Manifest hash, file count, access record"},
    {"module":"Image/subject identity","status":"completed_partial","evidence":"Byte, pixel, and perceptual identity; no patient/study IDs"},
    {"module":"Label integrity","status":"completed","evidence":"Contradictory exact and perceptual groups"},
    {"module":"Provenance","status":"completed_partial","evidence":"Folder source and native features; upstream provenance missing"},
    {"module":"Partition integrity","status":"completed_partial","evidence":"Image groups checked; patient/source independence unavailable"},
    {"module":"Shortcut controls","status":"completed" if reference_results.get("status")=="completed" else "not_evaluated","evidence":"Metadata-only endpoint prediction"},
    {"module":"Reliability evidence","status":"completed" if reference_results.get("status")=="completed" else "not_evaluated","evidence":"Discrimination, calibration, conformal, and referral"},
    {"module":"Claim gate","status":"completed","evidence":"Versioned claim-rule-evidence records"},
])
passport_modules.to_csv(TAB_DIR/"Table1_Passport_Modules.csv",index=False)

cxr_gate_table=pd.DataFrame([
    {"pipeline_state":"Raw compilation","files_retained":cxr_integrity["source_files"],"principal_evidence":f"{cxr_integrity['exact_conflict_files']} files in contradictory exact identities","claim_gate":"blocked_random_split"},
    {"pipeline_state":"Exact-conflict control","files_retained":len(unique_cxr),"principal_evidence":"Conflicts removed; consistent exact copies reduced","claim_gate":"conditional_pending_approximate_identity"},
    {"pipeline_state":"Exact + perceptual control","files_retained":cxr_integrity["final_clean_images"],"principal_evidence":f"{cxr_integrity['near_conflict_groups']} cross-label perceptual groups excluded","claim_gate":"eligible_group_partition"},
    {"pipeline_state":"Group-aware evaluation","files_retained":cxr_integrity["final_clean_images"],"principal_evidence":"No detected identity crosses reconstructed partitions","claim_gate":"supported_identity_level"},
    {"pipeline_state":"Shortcut control","files_retained":cxr_integrity["final_clean_images"],"principal_evidence":json.dumps(reference_results.get("metadata_shortcut",{})),"claim_gate":next(g["state"] for g in gates if g["claim_id"]=="C4")},
])
cxr_gate_table.to_csv(TAB_DIR/"Table3_CXR_Audit_Evidence_Gates.csv",index=False)
portability_table.loc[:,"schema_valid"]=True;portability_table.to_csv(TAB_DIR/"Table5_Cross_Dataset_Portability.csv",index=False)
log("Passport schema validated. Claim gates:\n"+pd.DataFrame([{**g,"model_scope":(g["model_scope"] or {}).get("model","")} for g in gates])[["claim_id","depends_on","state","effective_state","rule_id","model_scope"]].to_string(index=False))



## 12. Human-review candidates and optional agreement calculation

No human decision is generated by code. Reviewers should independently fill
`reviewer_1` and `reviewer_2` in the exported CSV using: identical,
near_identical, different, or indeterminate. On a later run, the notebook will
compute observed agreement and Cohen's kappa.



In [ ]:
review_candidates=pair_eval.loc[pair_eval.near_pred.eq(1)].copy()
review_candidates["phash_distance"]=[(int(lookup.loc[a,"phash"],16)^int(lookup.loc[b,"phash"],16)).bit_count()
    for a,b in zip(review_candidates.base_record_id,review_candidates.other_record_id)]
review_candidates["priority"]=(review_candidates.phash_distance-CONFIG["phash_radius"]).abs()
review_candidates=(review_candidates.sort_values(["priority","fault_type"]).groupby("fault_type",group_keys=False).head(8)
                   .head(60).reset_index(drop=True))
review_candidates["reviewer_1"]="";review_candidates["reviewer_2"]=""
pair_panel_dir=REVIEW_DIR/"Pairs";pair_panel_dir.mkdir(parents=True,exist_ok=True)
panel_paths=[]
for i,row in review_candidates.iterrows():
    a=lookup.loc[row.base_record_id];b=lookup.loc[row.other_record_id]
    with Image.open(a.path) as im_a,Image.open(b.path) as im_b:
        left=ImageOps.contain(im_a.convert("RGB"),(320,320));right=ImageOps.contain(im_b.convert("RGB"),(320,320))
        canvas=Image.new("RGB",(660,360),"white");canvas.paste(left,(0,20));canvas.paste(right,(340,20))
        panel=pair_panel_dir/f"pair_{i:03d}.png";canvas.save(panel)
    panel_paths.append(str(panel))
review_candidates["panel_path"]=panel_paths
review_file=REVIEW_DIR/"human_review_candidates.csv"
if not review_file.exists():review_candidates.to_csv(review_file,index=False)
completed_review=REVIEW_DIR/"human_review_completed.csv"
if completed_review.exists():
    reviewed=pd.read_csv(completed_review).dropna(subset=["reviewer_1","reviewer_2"])
    valid=reviewed.reviewer_1.ne("")&reviewed.reviewer_2.ne("");reviewed=reviewed.loc[valid]
    if len(reviewed):
        from sklearn.metrics import cohen_kappa_score
        human_review_summary={"status":"completed","n":len(reviewed),"observed_agreement":float((reviewed.reviewer_1==reviewed.reviewer_2).mean()),
            "cohen_kappa":float(cohen_kappa_score(reviewed.reviewer_1,reviewed.reviewer_2))}
    else:human_review_summary={"status":"not_evaluated","reason":"Completed file contains no paired decisions."}
else:
    human_review_summary={"status":"not_evaluated","reason":"Two independent reviewers must complete the exported candidate sheet."}
pd.DataFrame([human_review_summary]).to_csv(TAB_DIR/"Human_Review_Summary.csv",index=False)
log("Human review: "+json.dumps(human_review_summary))



## 13. Manuscript-ready figures

All figures are written as high-resolution PNG files without embedded captions.



In [ ]:
sns.set_theme(style="whitegrid",context="notebook")
palette={"blue":"#2F6B9A","green":"#3B8C6E","gold":"#D99A2B","red":"#B64949","purple":"#785A9B","gray":"#64748B"}

# Figure 1: passport architecture
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(14,6.2));ax.set_xlim(0,14);ax.set_ylim(0,7);ax.axis("off")
def pbox(x,y,w,h,title,lines,color):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.03,rounding_size=0.10",linewidth=1.4,edgecolor="#243447",facecolor=color))
    ax.text(x+.16,y+h-.30,title,fontsize=11,fontweight="bold",va="top",color="#17202A")
    ax.text(x+.16,y+h-.75,"\n".join(lines),fontsize=8.5,va="top",color="#243447",linespacing=1.25)
def parr(x1,y1,x2,y2):ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="-|>",mutation_scale=14,linewidth=1.4,color="#40566F"))
pbox(.35,2.2,2.1,2.5,"Dataset input",["Files and labels","Source metadata","Split assignments","Code + environment"],"#DCEBFA")
pbox(2.95,1.0,2.75,5.0,"Executable audits",["Identity + duplicates","Label conflicts","Provenance profile","Partition leakage","Shortcut controls","Calibration","Conformal + referral"],"#DDF3E4")
pbox(6.25,2.0,2.55,3.0,"Versioned evidence",["Raw measurements","Traceable identifiers","Thresholds + rules","Missingness states","Runtime + environment"],"#FFF1C7")
pbox(9.35,1.55,2.0,3.85,"Claim gate",["Supported","Conditional","Blocked","Not evaluated","","No composite score"],"#FAD7D7")
pbox(11.9,2.05,1.75,2.85,"Outputs",["passport.json","Audit tables","Human report","Execution summary"],"#E8DDF4")
for a,b in [((2.45,3.45),(2.95,3.45)),((5.7,3.45),(6.25,3.45)),((8.8,3.45),(9.35,3.45)),((11.35,3.45),(11.9,3.45))]:parr(*a,*b)
ax.text(7,6.55,"Imaging Dataset Reliability Passport",ha="center",fontsize=16,fontweight="bold",color="#17202A")
fig.tight_layout();fig.savefig(FIG_DIR/"Figure1_Passport_Architecture.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 2: controlled fault validation
plot=fault_metrics.melt(id_vars="fault_type",value_vars=["precision","recall","f1"],var_name="metric",value_name="value")
fig,axes=plt.subplots(1,2,figsize=(13,4.8),gridspec_kw={"width_ratios":[2.2,1]})
sns.barplot(data=plot,x="fault_type",y="value",hue="metric",ax=axes[0],palette=[palette["blue"],palette["green"],palette["gold"]])
axes[0].set(ylim=(0,1.05),xlabel="Fault family",ylabel="Performance",title="Controlled fault detection");axes[0].tick_params(axis="x",rotation=25)
sns.barplot(data=fault_metrics,x="fault_type",y="false_positive_rate",ax=axes[1],color=palette["red"])
axes[1].set(ylim=(0,max(.06,float(fault_metrics.false_positive_rate.max())*1.15)),xlabel="Fault family",ylabel="False-positive rate",title="False-positive analysis");axes[1].tick_params(axis="x",rotation=25)
fig.tight_layout();fig.savefig(FIG_DIR/"Figure2_Controlled_Fault_Validation.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 3: CXR audit and claim states
stage_plot=cxr_gate_table.copy();state_colors={"blocked_random_split":palette["red"],"conditional_pending_approximate_identity":palette["gold"],
    "eligible_group_partition":palette["green"],"supported_identity_level":palette["blue"],"blocked":palette["red"],"conditional":palette["gold"],
    "supported_in_evaluated_setting":palette["green"],"not_evaluated":palette["gray"]}
fig,ax=plt.subplots(figsize=(10,5));colors=[state_colors.get(s,palette["purple"]) for s in stage_plot.claim_gate]
bars=ax.bar(stage_plot.pipeline_state,stage_plot.files_retained,color=colors)
ax.bar_label(bars,fmt="%d",padding=3,fontsize=9);ax.set(ylabel="Files retained",xlabel="Pipeline state",title="Real-world CXR passport across audit states")
ax.tick_params(axis="x",rotation=22);fig.tight_layout();fig.savefig(FIG_DIR/"Figure3_CXR_Passport.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 4: model interpretation before and after the shortcut gate
if reference_results.get("status")=="completed":
    comparison=pd.DataFrame([{"model":m,"macro_f1":v["macro_f1"]} for m,v in reference_results["test_metrics"].items()]+[
        {"model":"Metadata control","macro_f1":reference_results["metadata_shortcut"]["macro_f1"]}])
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    sns.barplot(data=comparison,x="model",y="macro_f1",ax=axes[0],palette=[palette["blue"],palette["green"],palette["red"]])
    axes[0].set(ylim=(0,1),xlabel="Evidence source",ylabel="Macro-F1",title="Internal performance and negative control")
    reliability=pd.DataFrame([{"evidence":"Conformal coverage","value":reference_results["conformal"]["marginal_coverage"]},
        {"evidence":"Selective coverage","value":reference_results["selective"]["coverage"]},
        {"evidence":"Accepted accuracy","value":reference_results["selective"]["accepted_accuracy"]}])
    sns.barplot(data=reliability,x="evidence",y="value",ax=axes[1],color=palette["purple"])
    axes[1].axhline(1-CONFIG["conformal_alpha"],ls="--",color="black",lw=1);axes[1].set(ylim=(0,1.05),xlabel="Reliability evidence",ylabel="Value",title="Conformal and selective behavior");axes[1].tick_params(axis="x",rotation=20)
    fig.tight_layout();fig.savefig(FIG_DIR/"Figure4_Audit_Stages_Model_Interpretation.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 5: threshold stability and computational scaling
fig,axes=plt.subplots(1,2,figsize=(12,4.6))
for metric,color in [("precision",palette["blue"]),("recall",palette["green"]),("f1",palette["gold"])]:
    axes[0].plot(threshold_sensitivity.radius,threshold_sensitivity[metric],"o-",label=metric.title(),color=color)
axes[0].set(ylim=(0,1.05),xlabel="pHash Hamming radius",ylabel="Performance",title="Near-duplicate threshold sensitivity");axes[0].legend()
axes[1].plot(scaling.n_images,scaling.runtime_seconds,"o-",color=palette["purple"],label="Runtime")
ax2=axes[1].twinx();ax2.plot(scaling.n_images,scaling.peak_memory_mb,"s--",color=palette["red"],label="Peak memory")
axes[1].set(xlabel="Images",ylabel="Runtime (s)",title="Computational scaling");ax2.set_ylabel("Peak traced memory (MB)")
lines=axes[1].lines+ax2.lines;axes[1].legend(lines,[l.get_label() for l in lines],loc="upper left")
fig.tight_layout();fig.savefig(FIG_DIR/"Figure5_Threshold_Stability_Scalability.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)

# Figure 6: cross-dataset module portability
port_plot=portability_modules.copy();port_plot["completed"]=port_plot.status.str.startswith("completed").astype(int)
fig,ax=plt.subplots(figsize=(9,4.8));sns.barplot(data=port_plot,x="module",y="completed",ax=ax,color=palette["green"])
ax.set(ylim=(0,1.08),xlabel="Passport module",ylabel="Completion state",title="Cross-modality portability on PathMNIST",yticks=[0,1],yticklabels=["Not evaluated","Completed/partial"]);ax.tick_params(axis="x",rotation=28)
fig.tight_layout();fig.savefig(FIG_DIR/"Figure6_Cross_Dataset_Portability.png",dpi=300,bbox_inches="tight");plt.show();plt.close(fig)



## 14. Manuscript-ready summaries and result archive



In [ ]:
def fmt(x,d=4):return "not evaluated" if x is None or (isinstance(x,float) and np.isnan(x)) else f"{x:.{d}f}"
snippets=["# Manuscript replacement snippets","",
"## Passport schema and implementation",
f"The frozen passport schema was version {CONFIG['schema_version']} and contained {schema_summary['declared_fields']} declared fields across eight evidence modules. The generated passport passed JSON Schema validation and produced {len(gates)} traceable claim-gate records.","",
"## Controlled fault detection"]
for _,r in fault_metrics.iterrows():
    snippets.append(f"- {r.fault_type}: precision {r.precision:.4f}, recall {r.recall:.4f} ({int(r.tp)}/{int(r.tp+r.fn)}), F1 {r.f1:.4f}, false positives {int(r.fp)}/{int(r.fp+r.tn)}.")
snippets += ["",f"Near-identity strata (radius {CONFIG['phash_radius']}): minimum stratum {min_stratum} = {min_stratum_recall:.4f}; design-based SE {design_se:.4f}.",
             f"Direct pairwise recall {direct_vs_component['direct_pairwise_recall']:.4f} vs same-component {direct_vs_component['same_component_recall']:.4f}; transitive fraction {direct_vs_component['transitive_fraction']:.4f}.",
             f"Two-stage detector: {json.dumps(two_stage_results)}",
             f"Fine-tuned model: {json.dumps(reference_results.get('finetuned',{}).get('test_metrics','not evaluated'))}"]
snippets += ["","## Real-world CXR passport",
f"The regenerated audit contained {cxr_integrity['source_files']:,} source files, {cxr_integrity['exact_conflict_files']:,} files in contradictory exact identities, {cxr_integrity['near_conflict_groups']:,} cross-label perceptual groups comprising {cxr_integrity['near_conflict_images']:,} images, and {cxr_integrity['final_clean_images']:,} retained images.","",
"## Reproducibility and portability",
f"Repeated execution produced exact-output agreement {float(digest_a==digest_b):.4f} and perceptual-group adjusted Rand index {ari:.4f}. Independent-environment comparison was {cross_status}.",
f"The PathMNIST adapter completed {completed} of {len(portability_modules)} modules without core-code modification; patient/study identity and image-model reliability remained unavailable.","",
"## Human review",human_review_summary.get("reason",json.dumps(human_review_summary))]
(ROOT/"Manuscript_Replacement_Snippets.md").write_text("\n".join(snippets),encoding="utf-8")

summary=["IMAGING DATASET RELIABILITY PASSPORT — RUN COMPLETE",f"Output root: {ROOT}",f"Schema valid: True (version {CONFIG['schema_version']})",
    f"Controlled faults: minimum recall={fault_metrics.recall.min():.4f}; maximum FPR={fault_metrics.false_positive_rate.max():.4f}",
    f"CXR: source={cxr_integrity['source_files']:,}; retained={cxr_integrity['final_clean_images']:,}",
    f"PathMNIST: files={len(path_audit):,}; modules completed={completed}/{len(portability_modules)}",
    f"Repeated exact agreement={float(digest_a==digest_b):.4f}; perceptual-group ARI={ari:.4f}",f"Second environment: {cross_status}",
    f"Human review: {human_review_summary['status']}","","CLAIM GATES:"]
summary += [f"- {g['claim_id']}: {g['state']} — {g['rationale']}" for g in gates]
summary += ["","INTERPRETATION BOUNDARIES:","- The CXR model evaluation remains internal to a public compilation.",
    "- Patient and institution independence cannot be established without identifiers.","- The passport is an evidence-governance artifact, not a clinical or regulatory certificate."]
LOG_PATH.write_text("\n".join(summary),encoding="utf-8");print("\n".join(summary))

archive_path=ROOT.parent/"IMU_Reliability_Passport_Results.zip"
with zipfile.ZipFile(archive_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for path in ROOT.rglob("*"):
        if path.is_file() and "FaultBenchmark/images" not in str(path) and "Portability_PathMNIST" not in str(path):
            archive.write(path,path.relative_to(ROOT))
print("Results archive:",archive_path)
if CONFIG["auto_download_archive"] and IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))



## Interpretation checklist

- Controlled-fault results are objective only for the injected transformations.
- Natural CXR findings must be described as regenerated reference evidence.
- A second environment is not claimed unless a distinct fingerprint bundle exists.
- Human agreement is not claimed until two reviewers complete the candidate CSV.
- PathMNIST supports software and cross-modality portability, not clinical generalization.
- The generated passport restricts claims; it does not certify clinical safety.
